# BDB27 — leverage × ceiling pipeline (full run)

**Does situational leverage change how much of the ceiling receivers use on max-effort routes?**
(WR primary, TE replication) — `TASK.md` is the binding spec.

This notebook is the *runnable* deliverable: it opens, you set two things in the
**Configuration** cell below (`MODE` and — only if needed — `DATA_DIR`), then **Run All**.
No terminal required.

It drives `src/` in dependency order: the **current LC chain first**, then the preserved
**ceiling-v1 chain** (parked under `src/legacy_ceiling_v1/` by D59; it still runs from where it
lives, and §5–§12 read its outputs).

```
LC chain (D59–D63) : 01_audit → 02_combine_ceiling → 00_thresholds → 03_route_depth
ceiling-v1 (D27–D50): 02_combine_envelope → 06_security → 03_game_demand → 07_security_models →
                      10_dose_response → 00b_external_proxies → 04_models → 05_validate
```

and renders the **≤5 writeup figures** into `outputs/figures/`.

**Where the current answer lives: §13.** It reads the LC outputs off disk — break coverage, coverage
by route family, and the D62/D63 keep-or-fall-back rule for the continuous depth control. No number
in it is typed by hand. §8–§12 are the *retired* ceiling-v1 results and are labelled as such.

Two similarly named stages are **different stages**, not a rename:

| LC (current) | ceiling-v1 (retired, preserved) |
|---|---|
| `02_combine_ceiling` → `outputs/ceiling/lc/` | `02_combine_envelope` → `outputs/envelope/` |
| `03_route_depth` → `outputs/game/lc/route_depth/` | `03_game_demand` → `outputs/game/` |

### Reading the labels

* `sample` mode outputs are stamped **`UNVALIDATED SAMPLE OUTPUT`** — plumbing, not findings.
* `full` mode outputs are stamped **`FULL`** — the real analysis.
* synthetic-fixture outputs are stamped **`SYNTHETIC FIXTURE (not a finding)`** — they prove the
  plumbing, nothing about the NFL.

### D38 — the retired security spine (§5–§12)

Analysis A (the old "did the market reward speed?" check) was retired as a gate; the ceiling-v1
spine was **incentive exposure** (`moved_down` vs `high_secure`, assigned from pre-Combine EDP only).
That work is **not** the current question (D59). Its sections are kept because their code and
outputs still run, and deleting a working audit trail is not an improvement; read them as history.


## 1. Configuration

Edit only this cell. `MODE="sample"` runs everything the Pi's partial data allows and clearly flags
the game-side stages as `PENDING FULL RUN`. Set `MODE="full"` once the full
`game_tracking_{2023,24,25}.csv` files are available.

`CLEAN=True` is destructive but **inert unless `CLEAN_CONFIRM="DELETE"`**. When armed it deletes every
stage output under `outputs/` **and the LC stage checkpoints under `data/interim/checkpoints/`**, so the
next run is a genuine rebuild rather than a skip against a checkpoint whose outputs no longer exist.
Leave it `False` to reuse checkpoints. `FORCE_STAGES` rebuilds only the named stages
(e.g. `["03_route_depth"]`, or `["02_combine_envelope"]` for the ceiling-v1 stage); `FORCE=True`
forces every stage.

`RUN_ONLY=[]` (default) runs the whole pipeline in `STAGE_ORDER`. Set it to a list of stage names to
run **only** those stages, in `STAGE_ORDER` order, reusing every other checkpoint:
`RUN_ONLY=["03_route_depth"]` re-runs just the route-depth stage after a detector change — minutes
instead of a full pass. Skipped stages are printed with their on-disk state (never as failures).

**Stage names are resolved against `src/` first, then `src/legacy_ceiling_v1/`.** A name that exists
in both trees (`00_thresholds`) always means the **current** module; ceiling-v1 names that no longer
exist in `src/` (`02_combine_envelope`, `03_game_demand`, …) fall through to the parked copy.

### D49 — season scope (ceiling-v1 specification only)

The ceiling-v1 primary specification is **REG + POST** games only. It is applied at that chain's
model stage; the LC chain does not read `game.scope`. `SEASON_SCOPE_PRIMARY`,
`SEASON_SCOPE_SENSITIVITY` and `SEASON_SCOPE_ACTIVE` below set the scope written into the run config,
and §11 runs the sensitivity scope in-notebook.

**Fingerprints / stale artefacts.** Every stage checkpoint records a config fingerprint. A missing or
changed fingerprint is a **mismatch**, so the first run after a config change rejects those markers and
rebuilds those stages. §3 prints the rejection reason per stage and a clear
`artefacts stale — ONE FULL REBUILD REQUIRED` banner.


In [ ]:
# ===================== CONFIG — edit me (nothing else) =====================
# Game-side per-frame speed is despiked at game.speed_cap_yds (12.5 yd/s; frame-level data-QC despike, D35) — re-run the game chain (FORCE/CLEAN) after a QC change.
MODE               = "sample"   # "sample" | "full"
DATA_DIR           = None       # None = auto-detect (/kaggle/input/... else <repo>/data/raw)
FORCE              = False      # True = force EVERY stage (global override of FORCE_STAGES)
FORCE_STAGES       = []         # stage names to rebuild, e.g. ["02_combine_envelope"]; [] = reuse checkpoints
RUN_ONLY           = []         # [] = every stage in STAGE_ORDER. Non-empty = run ONLY these stages,
                                #   in STAGE_ORDER order, reusing all other checkpoints:
                                #   ["07_security_models"] = stage-07-only re-run (seconds, not hours).
                                #   Stages not listed are SKIPPED, never rebuilt and never marked failed.
CLEAN              = False      # True deletes every stage output under outputs/ (including the Combine
                                #   envelope refit) — destructive. Requires CLEAN_CONFIRM == "DELETE".
CLEAN_CONFIRM      = ""         # safety interlock: CLEAN=True is ignored unless this is exactly "DELETE"
# ---- D49 season scope: ONE build serves both; the scope is applied at the model stage ----
SEASON_SCOPE_PRIMARY     = ["REG", "POST"]        # new primary specification (no preseason)
SEASON_SCOPE_SENSITIVITY = ["REG", "POST", "PRE"] # all-games version, retained as the sensitivity
SEASON_SCOPE_ACTIVE      = "primary"              # "primary" | "sensitivity" (which §3 models fit)
SEASON_PANEL_ACTIVE      = "all_seasons"          # "all_seasons" | "rookie_only"
# D49 WARNING: this commit bumps pipeline_version and adds config fingerprints. Every existing
# stage checkpoint is rejected ONCE; the first full run is a one-off full rebuild (no resume).
RUN_SYNTHETIC_DEMO = True       # exercise the game-side path on the committed fixtures
SYNTHETIC_REGIMES  = ("binding", "null")            # ceiling fixture (04/05)
SECURITY_REGIMES   = ("planted", "null", "selection")  # security fixture (07)
SHOW_SYNTHETIC_PREVIEW = False  # §5/§8: True renders the synthetic LAYOUT PREVIEW (never a finding).
                                #   False = synthetic numbers are withheld wherever a result could be read.
SHOW_FIGURES       = True       # display figures inline
FIG_DPI            = 150
assert MODE in ("sample", "full"), MODE
assert isinstance(FORCE_STAGES, (list, tuple)), FORCE_STAGES
assert all(isinstance(s, str) and s for s in FORCE_STAGES), FORCE_STAGES
assert isinstance(RUN_ONLY, (list, tuple)), RUN_ONLY
assert all(isinstance(s, str) and s for s in RUN_ONLY), RUN_ONLY
assert isinstance(CLEAN_CONFIRM, str), CLEAN_CONFIRM
assert SEASON_SCOPE_ACTIVE in ("primary", "sensitivity"), SEASON_SCOPE_ACTIVE
assert SEASON_PANEL_ACTIVE in ("all_seasons", "rookie_only"), SEASON_PANEL_ACTIVE
assert set(SEASON_SCOPE_PRIMARY) <= set(SEASON_SCOPE_SENSITIVITY), "primary must be a subset"
assert all(t in ("PRE", "REG", "POST") for t in SEASON_SCOPE_SENSITIVITY), SEASON_SCOPE_SENSITIVITY
assert isinstance(SHOW_SYNTHETIC_PREVIEW, bool), SHOW_SYNTHETIC_PREVIEW
_EFFECTIVE_FORCE = ("ALL STAGES (FORCE=True)" if FORCE
                    else (list(FORCE_STAGES) or "none — reuse checkpoints"))
_CLEAN_ARMED = (CLEAN is True) and (CLEAN_CONFIRM == "DELETE")
print(f"effective --force : {_EFFECTIVE_FORCE}")
print(f"RUN_ONLY          : {list(RUN_ONLY) or 'none — run every stage in STAGE_ORDER'}")
print("CLEAN={} · CLEAN_CONFIRM={} — {}".format(
    CLEAN, "set" if CLEAN_CONFIRM else "empty",
    "ARMED: every stage output under outputs/ will be deleted" if _CLEAN_ARMED
    else ("IGNORED: set CLEAN_CONFIRM='DELETE' to confirm" if CLEAN
          else "existing stage outputs are preserved")))
# ==========================================================================

## 2. Bootstrap — locate repo, data, write the run config

No commands to type: this cell finds the repository, picks the data folder, writes a scratch
run-config (`bdb27_run/config.yaml`) that points **both** engines at the data, and prints the resolved
paths and mode.

It puts `src/` on the import path for the LC stages and `src/legacy_ceiling_v1/` behind it for the
preserved ceiling-v1 stages, then imports two engines: `lc_common` (current task) and `common`
(ceiling-v1). The checkout guard is the ceiling-v1 one; it blocks only when a required capability is
missing or the checkout is behind its upstream.


In [ ]:
import importlib, io, json, sys, time, traceback, shutil
from contextlib import redirect_stdout, redirect_stderr
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

# ---- locate the repo root (config.yaml + src/lc_common.py) -----------------
# D59 parked the ceiling-v1 engine under src/legacy_ceiling_v1/ and gave the current task its own
# engine (src/lc_common.py). The notebook needs BOTH: the LC chain is current work; the ceiling-v1
# stages are preserved (D63) and §5–§12 read their outputs.
def find_repo_root():
    c = Path.cwd().resolve()
    for p in [c, *c.parents]:
        if (p / "config.yaml").exists() and (p / "src" / "lc_common.py").exists():
            return p
    raise RuntimeError("Cannot locate the BDB27 repo root (need config.yaml + src/lc_common.py) "
                       "— open this notebook from inside the BDB27 checkout.")

REPO   = find_repo_root()
SRC    = REPO / "src"
LEGACY = SRC / "legacy_ceiling_v1"

# src/ is inserted last at position 0, so it is searched FIRST: a stage name present in both trees
# (00_thresholds) must resolve to the CURRENT module, never to a parked copy.
for _p in (LEGACY, SRC):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import lc_common as LC          # current-task engine: config, paths, provenance, checkpoints
import stage_runner             # D46: reload stages from disk + crash reports
# D46: snapshot BEFORE this cell imports anything from src/. `import` returns the sys.modules copy,
# so a module already present here may be older than the file on disk.
PRELOADED_AT_CELL_START = stage_runner.preloaded_modules()
import common                   # ceiling-v1 engine (display sections + legacy stage arguments)
import run_diagnostics          # stale-checkout guard + honest failure helpers

# LC stages write under outputs/ but keep their checkpoints in data/interim/checkpoints/. Tell the
# (ceiling-v1) status reader where each LC stage writes, so a RUN_ONLY skip is reported from the
# artefacts §13 actually reads.
run_diagnostics.STAGE_OUTPUTS.update({
    "01_audit":           ("audit/lc", ("decision.json",)),
    "02_combine_ceiling": ("ceiling/lc", ("player_ceiling.csv", "summary.json")),
    "00_thresholds":      ("", ("thresholds.json",)),
    "03_route_depth":     ("game/lc/route_depth", ("route_depth.csv", "route_depth_summary.json")),
    "04_efforts":         ("game/lc/efforts", ("efforts.csv", "efforts_summary.json")),
    "05_models":          ("game/lc/models", ("model_result.json", "model_coefficients.csv")),
    "06_validation":      ("game/lc/validation", ("validation_result.json", "SUMMARY.md")),
    "07_phase4":          ("game/lc/phase4", ("phase4_result.json", "SUMMARY.md")),
})

CHECKOUT = run_diagnostics.evaluate_checkout(REPO)
if not CHECKOUT["state"]["git_available"]:
    print("checkout : git unavailable — continuing without a stale-code check")
else:
    for line in CHECKOUT["header_lines"]:
        print(f"checkout : {line}")
if CHECKOUT["blocked"]:
    print("!" * 88)
    print(CHECKOUT["stop_message"])
    print("pipeline will NOT execute until the checkout is updated")
    print("!" * 88)

# ---- D46: is this kernel holding code that is older than the files on disk? --
CODE_STATE = stage_runner.code_state_report(REPO, modules=tuple(PRELOADED_AT_CELL_START))
if CODE_STATE["preloaded"]:
    print("!" * 88)
    print(f"CODE STATE: {len(CODE_STATE['preloaded'])} pipeline module(s) are ALREADY LOADED in this kernel")
    print("  — they may be OLDER than the files on disk (a pull does not refresh sys.modules):")
    for line in CODE_STATE["warnings"]:
        print(f"  !! {line}")
    print("  §3 reloads every stage module from disk, so the run below still uses the current code.")
    print("  (Restarting the kernel is the belt-and-braces option.)")
    print("!" * 88)
else:
    print("code state     : no pipeline module preloaded — each stage imports src/ from disk")

# bind the logger to the real stderr BEFORE any redirect, so stage logs stay visible
common.get_logger()
CFG_REPO = common.load_config(str(REPO / "config.yaml"))     # ceiling-v1 blocks (dose, external, game)
LC_CFG   = LC.lc(CFG_REPO)                                   # the current-task block (bdb27_lc)

# ---- pick the data directory ----------------------------------------------
def has_combine(d):
    return Path(d).is_dir() and (Path(d) / "combine_tracking.csv").exists()

def detect_data_dir(override):
    if override:
        p = Path(override).expanduser().resolve()
        if has_combine(p):
            return p
        hits = sorted(p.rglob("combine_tracking.csv"))
        return hits[0].parent if hits else p
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        hits = sorted(kaggle.rglob("combine_tracking.csv"))
        if hits:
            return hits[0].parent
    return REPO / "data" / "raw"

DATA = detect_data_dir(DATA_DIR)

# ---- scratch run-config (override only the data path; everything else stays default)
RUN_DIR = REPO / "bdb27_run"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_CFG_PATH = RUN_DIR / "config.yaml"
run_cfg = json.loads(json.dumps(CFG_REPO))          # deep copy (config is JSON-safe)
run_cfg.pop("_config_path", None)
# D49: the notebook knobs override the season-scope block (default REG+POST primary).
run_cfg["game"]["scope"] = {
    "primary": list(SEASON_SCOPE_PRIMARY),
    "sensitivity": list(SEASON_SCOPE_SENSITIVITY),
    "active": SEASON_SCOPE_ACTIVE,
    "active_panel": SEASON_PANEL_ACTIVE,
    "panels": ["all_seasons", "rookie_only"],
}
run_cfg["paths"]["raw"] = str(DATA)                 # absolute -> resolve() passes it through
run_cfg["bdb27_lc"]["paths"]["raw"] = str(DATA)     # D63: the LC stages resolve from this block
RUN_CFG_PATH.write_text(yaml.safe_dump(run_cfg, sort_keys=False), encoding="utf-8")
RUN_CFG = LC.load_config(str(RUN_CFG_PATH))

def _abs_path(p):
    p = Path(p)
    return p if p.is_absolute() else (REPO / p)

OUTPUTS = _abs_path(RUN_CFG["bdb27_lc"]["paths"]["outputs"])
FIGDIR  = _abs_path(RUN_CFG["bdb27_lc"]["paths"]["figures"])
LC_CKPT = _abs_path(RUN_CFG["bdb27_lc"]["paths"]["checkpoints"])
FIGDIR.mkdir(parents=True, exist_ok=True)

# ---- which raw files are present? -----------------------------------------
RAW_FILES = list(dict.fromkeys([
    "combine_tracking.csv", "combine_results.csv", "players.csv", "games.csv", "player_play.csv",
    *LC_CFG["files"]["game_tracking_full"],
    LC_CFG["files"]["game_tracking_sample"],
]))
present = {f: (DATA / f).exists() for f in RAW_FILES}
FULL_TRACKING = [f for f in LC_CFG["files"]["game_tracking_full"] if not (DATA / f).exists()]
full_game_ok = not FULL_TRACKING
combine_ok   = present["combine_tracking.csv"] and present["combine_results.csv"]

# CLEAN is destructive: it runs ONLY when CLEAN_CONFIRM == "DELETE". Otherwise nothing is deleted.
_CLEAN_DIRS = ("audit", "envelope", "ceiling", "game", "models", "validate", "security",
               "security_models")
_CLEAN_FILES = ("thresholds.json", "run_log.csv")

if CLEAN is True and CLEAN_CONFIRM == "DELETE":
    print("CLEAN=True confirmed — these outputs will be removed:")
    for d in _CLEAN_DIRS:
        p = OUTPUTS / d
        n = sum(1 for f in p.rglob("*") if f.is_file()) if p.exists() else 0
        print(f"  {p.relative_to(REPO)}/  ({n} files)")
    for fn in _CLEAN_FILES:
        p = OUTPUTS / fn
        print(f"  {p.relative_to(REPO)}  ({1 if p.exists() else 0} files)")
    print(f"  {FIGDIR.relative_to(REPO)}/  ({sum(1 for _ in FIGDIR.glob('*.png'))} PNGs)")
    n_ckpt = len(list(LC_CKPT.glob("*.json"))) if LC_CKPT.exists() else 0
    print(f"  {LC_CKPT.relative_to(REPO)}/  ({n_ckpt} LC stage checkpoints)")
    for d in _CLEAN_DIRS:
        shutil.rmtree(OUTPUTS / d, ignore_errors=True)
    for f in _CLEAN_FILES:
        try: (OUTPUTS / f).unlink()
        except FileNotFoundError: pass
    for f in FIGDIR.glob("*.png"):
        f.unlink()
    # D63: without this, a cleaned LC stage would report "up to date" from a checkpoint whose
    # outputs no longer exist. CLEAN promises a rebuild; make it one.
    if LC_CKPT.exists():
        for f in LC_CKPT.glob("*.json"):
            f.unlink()
elif CLEAN is True:
    print("CLEAN=True ignored: set CLEAN_CONFIRM='DELETE' to confirm")
    print("Existing stage outputs are preserved.")

# ---- shared output readers (used by the run + figure cells) -----------------
def out_path(*parts):
    return OUTPUTS.joinpath(*parts)

def read_csv(path, **kw):
    path = Path(path)
    return pd.read_csv(path, **kw) if path.exists() else pd.DataFrame()

def read_json(path):
    """Parsed JSON or None — never raises (a half-written file is 'absent', not a traceback)."""
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return None

def prov_of(df, fallback="UNVALIDATED SAMPLE OUTPUT"):
    if len(df) and "provenance" in df.columns:
        return str(df["provenance"].iloc[0])
    return fallback

def prov_of_json(path):
    """Provenance recorded inside a JSON artefact, or '' when it is absent/unreadable."""
    return str((read_json(path) or {}).get("provenance", ""))

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def is_true(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])

def rel(path):
    """Repo-relative display path (never raises for a temp/copy root)."""
    try:
        return str(Path(path).relative_to(REPO))
    except ValueError:
        return str(path)

def stage_status_text(name):
    """Status from RUN_RESULTS, else from the stage's own on-disk checkpoint (D45).

    A stage skipped by RUN_ONLY / not re-run in this session must never read as a failure: we
    report the checkpoint/output state that §8 is actually reading.
    """
    results = globals().get("RUN_RESULTS") or []
    for r in reversed(results):
        if r.get("stage") == name:
            txt = str(r.get("status") or ("OK" if r.get("ok") else "FAILED"))
            if r.get("error"):
                txt += f" · error: {r['error']}"
            return txt
    return run_diagnostics.stage_status_on_disk(name, OUTPUTS)

print(f"repo root      : {REPO}")
print(f"data dir       : {DATA}")
print(f"mode           : {MODE}   (force={FORCE}, clean={CLEAN})")
print(f"Combine files  : {'present' if combine_ok else 'MISSING'}")
print(f"full game files: {'present' if full_game_ok else 'ABSENT -> game-side stages are PENDING FULL RUN'}")
if FULL_TRACKING:
    for f in FULL_TRACKING:
        print(f"                 missing: {f}")
print(f"run config     : {RUN_CFG_PATH.relative_to(REPO)}")
print(f"engines        : lc_common (LC chain) + src/legacy_ceiling_v1 (ceiling-v1 chain, preserved)")
print(f"season scope   : primary {'+'.join(SEASON_SCOPE_PRIMARY)} "
      f"(active={SEASON_SCOPE_ACTIVE}); sensitivity {'+'.join(SEASON_SCOPE_SENSITIVITY)}  [ceiling-v1 only]")
print(f"pipeline ver.  : {RUN_CFG.get('pipeline_version')} — a changed fingerprint forces a rebuild")
print(f"outputs        : {OUTPUTS.relative_to(REPO)}/   figures -> {FIGDIR.relative_to(REPO)}/")


## 3. Run the pipeline in order

Each stage is invoked through its own entry point (so its `--mode` / `--force` / checkpoint
behaviour and the checkout guard all apply exactly as on the command line), with the scratch
run-config passed explicitly (`--config bdb27_run/config.yaml`). A stage that needs data this machine
does not have is caught and reported as `PENDING FULL RUN` rather than crashing the notebook.

`STAGE_ORDER` is the **current LC chain first, then the preserved ceiling-v1 chain**:

* **LC (D59–D63)** — `01_audit → 02_combine_ceiling → 00_thresholds → 03_route_depth`.
  The order comes from `TASK.md`'s declared phase order; no LC stage consumes another's output file,
  and `03_route_depth` is the only one that asserts the run's provenance. `02_combine_ceiling` is
  *not* game-side: the Combine file is complete, so it runs for real in both modes.
* **ceiling-v1 (D27–D50, retired question)** — `02_combine_envelope → 06_security → 03_game_demand →
  07_security_models → 10_dose_response → 00b_external_proxies → 04_models → 05_validate`.
  Preserved, and it still runs. Its stages are resolved from `src/legacy_ceiling_v1/` only where
  `src/` has no module of that name.

Repeated runs reuse each stage's checkpoint (LC stages checkpoint under
`data/interim/checkpoints/`, ceiling-v1 stages under `outputs/<stage>/_DONE`). Name a stage in
`FORCE_STAGES` to rebuild only that stage; `FORCE=True` is the all-stages override.

Every stage row ends with `code reloaded from disk` (D46): `run_stage()` invalidates the import
caches and `importlib.reload`s the module (and, for LC stages, its `lc_*` helpers) before calling
`main()`, so re-running this cell after a `git pull` executes the file that is on disk now — not the
copy cached in the kernel. If a stage crashes, the **full** traceback is written to a crash file and
the path is printed.

`RUN_ONLY` (Configuration) restricts the loop to the named stages, still in `STAGE_ORDER` order.
Every other stage is skipped and its existing checkpoint is reused; the skipped stages are printed
with their on-disk state, never as `FAILED` or `PENDING`.


In [ ]:
# ---- D63: ONE run order, TWO chains ---------------------------------------------------------
# LC chain (current task: TASK.md + D59–D63). Order read from the code on disk, not assumed:
#   01_audit           Combine + players/player_play (+ sample tracking) -> outputs/audit/lc/
#   02_combine_ceiling Combine tracking/results/players only             -> outputs/ceiling/lc/
#   00_thresholds      the frozen record -> outputs/thresholds.json (refuses FULL on sample input)
#   03_route_depth     players/player_play + tracking (parquet mirrors)  -> outputs/game/lc/route_depth/
#   04_efforts         route snaps + tracking + 03 depth table           -> outputs/game/lc/efforts/
#   05_models          efforts table only                                -> outputs/game/lc/models/
#   06_validation      Phase 5 checks on the primary fit                 -> outputs/game/lc/validation/
#   07_phase4          Phase 4 remainder (D67) on the same effort table  -> outputs/game/lc/phase4/
# 04 reads 03's route-depth table and 05 reads 04's efforts table; otherwise the order is TASK.md's
# declared phase order (audit -> ceiling -> thresholds -> depth -> efforts -> model), and 03-05 each
# assert the run's provenance.
# Keeping that order is what makes one Run All reproduce the command-line sequence in README_RUN.md.
LC_STAGE_ORDER = ["01_audit", "02_combine_ceiling", "00_thresholds", "03_route_depth",
                    "04_efforts", "05_models", "06_validation", "07_phase4"]

# ceiling-v1 chain: retired question, preserved code (parked in src/legacy_ceiling_v1/ by D59).
# It still runs from where it lives; §5–§12 read its outputs. Named 02_combine_envelope /
# 03_game_demand to distinguish it from the LC 02_combine_ceiling / 03_route_depth — different
# stages, different outputs, not a rename.
LEGACY_STAGE_ORDER = ["02_combine_envelope", "06_security", "03_game_demand", "07_security_models",
                      "10_dose_response", "00b_external_proxies", "04_models", "05_validate"]

STAGE_ORDER = LC_STAGE_ORDER + LEGACY_STAGE_ORDER
LC_STAGES   = set(LC_STAGE_ORDER)
# A failure in a full-data stage is PENDING FULL RUN, not FAILED. 02_combine_ceiling is NOT
# game-side (the Combine file is complete), so a failure there must show as FAILED.
GAME_SIDE = {"01_audit", "00_thresholds", "03_route_depth", "04_efforts", "05_models",
             "06_validation", "07_phase4",
             "03_game_demand", "07_security_models", "10_dose_response",
             "00b_external_proxies", "04_models", "05_validate"}
CHECKOUT_BLOCKED = bool(CHECKOUT.get("blocked"))

# LC stages reload their own helpers, so a `git pull` cannot leave a stale `lc_*` module in the
# kernel (D46). The ceiling-v1 helpers are reloaded by stage_runner itself.
LC_HELPERS = ("lc_common", "lc_route_depth", "ceiling_core", "lc_efforts",
               "lc_models", "lc_validation", "lc_phase4")

def run_stage(module_name, extra_args):
    """Run one src/ stage in-process, ALWAYS from the code on disk (D46).

    stage_runner reloads the stage (and its already-loaded helpers) before calling main(), so
    re-running this cell after a `git pull` executes the new file, not the kernel's cached copy.
    On a crash the full traceback goes to outputs/<stage>/crash_<stamp>.txt (path in the result).

    Both engines read the scratch run-config explicitly, so the notebook's data path and knobs are
    what the stage sees — never a stale default.
    """
    args = [str(a) for a in extra_args]
    if "--config" not in args:
        args += ["--config", str(RUN_CFG_PATH)]
    if module_name in LC_STAGES:
        return stage_runner.run_stage(module_name, args, outputs_root=OUTPUTS, helpers=LC_HELPERS)
    return stage_runner.run_stage(module_name, args, outputs_root=OUTPUTS)

SUMMARY_FILE = {"03_game_demand": ("game", "game_demand_summary.csv"),
                "07_security_models": ("security_models", "security_models_summary.csv"),
                "10_dose_response": ("security_models/dose", "dose_summary.csv"),
                "04_models":      ("models", "models_summary.csv"),
                "05_validate":    ("validate", "validation_summary.csv")}

def stage_output_status(name):
    """Honest per-stage status from the stage's OWN outputs (PENDING vs OK vs UNVALIDATED)."""
    try:
        # ---- LC chain (current task): status read from the stage's own JSON, never assumed ----
        if name == "01_audit":
            p = out_path("audit", "lc", "decision.json")
            rec = read_json(p)
            if rec is None:
                return "PENDING FULL RUN (no outputs/audit/lc/decision.json)"
            pending = ", ".join(rec.get("pending_full_run") or []) or "none"
            return f"{rec.get('verdict', '?')} (LC audit; pending full run: {pending})"
        if name == "02_combine_ceiling":
            prov = prov_of_json(out_path("ceiling", "lc", "summary.json"))
            if not prov:
                return "PENDING FULL RUN (no outputs/ceiling/lc/summary.json)"
            return "OK (FULL)" if prov == "FULL" else "OK (sample)"
        if name == "03_route_depth":
            p = out_path("game", "lc", "route_depth", "route_depth_summary.json")
            rec = read_json(p)
            if rec is None:
                return "PENDING FULL RUN (no outputs/game/lc/route_depth/)"
            cov = rec.get("coverage") or {}
            tag = "OK (FULL)" if str(rec.get("provenance", "")) == "FULL" else "OK (sample)"
            return (f"{tag} · {cov.get('n_break_found', 0):,} breaks / "
                    f"{cov.get('n_route_snaps', 0):,} route snaps")
        if name == "04_efforts":
            rec = read_json(out_path("game", "lc", "efforts", "efforts_summary.json"))
            if rec is None:
                return "PENDING FULL RUN (no outputs/game/lc/efforts/)"
            tag = "OK (FULL)" if str(rec.get("provenance", "")) == "FULL" else "OK (sample)"
            return (f"{tag} · {rec.get('n_efforts', 0):,} efforts / "
                    f"{rec.get('n_players', 0)} players")
        if name == "05_models":
            rec = read_json(out_path("game", "lc", "models", "model_result.json"))
            if rec is None:
                return "PENDING FULL RUN (no outputs/game/lc/models/)"
            tag = "OK (FULL)" if str(rec.get("provenance", "")) == "FULL" else "OK (sample)"
            b = rec.get("beta")
            btxt = f"{b:.4f}" if isinstance(b, (int, float)) else "NA"
            return f"{tag} · fallback={rec.get('fallback_used')} · beta={btxt}"
        if name == "06_validation":
            rec = read_json(out_path("game", "lc", "validation", "validation_result.json"))
            if rec is None:
                return "PENDING FULL RUN (no outputs/game/lc/validation/)"
            tag = "OK (FULL)" if str(rec.get("provenance", "")) == "FULL" else "OK (sample)"
            schemes = ((rec.get("permutation") or {}).get("schemes") or {})
            pg = (schemes.get("player_game") or {}).get("p_value")
            pgtxt = f"{pg:.3f}" if isinstance(pg, (int, float)) else "NA"
            verdict = (rec.get("equivalence") or {}).get("verdict", "NA")
            return f"{tag} · p(player_game)={pgtxt} · {verdict}"
        if name == "07_phase4":
            rec = read_json(out_path("game", "lc", "phase4", "phase4_result.json"))
            if rec is None:
                return "PENDING FULL RUN (no outputs/game/lc/phase4/)"
            tag = "OK (FULL)" if str(rec.get("provenance", "")) == "FULL" else "OK (sample)"
            matched = rec.get("matched_design") or {}
            dose = rec.get("dose_response") or {}
            slopes = rec.get("per_player_slopes") or {}
            n_not_run = len(rec.get("checks_not_run") or [])
            return (f"{tag} · matched {matched.get('status')} ({matched.get('n_pairs')} pairs) · "
                    f"dose monotone={dose.get('monotone')} · slopes={slopes.get('status')} · "
                    f"{n_not_run} NOT RUN")
        if name == "00_thresholds":
            p = out_path("thresholds.json")
            prov = json.loads(p.read_text())["provenance"] if p.exists() else None
            return "OK" if prov == "FULL" else f"PENDING FULL RUN (thresholds={prov})"
        if name == "01_audit":
            return "OK" if MODE == "full" else "PARTIAL (game-side c/h/i PENDING FULL RUN)"
        if name == "02_combine_envelope":
            p = out_path("envelope", "_DONE")
            m = json.loads(p.read_text()).get("mode") if p.exists() else None
            return "OK (FULL)" if m == "full" else "OK (sample)"
        if name == "06_security":
            p = out_path("security", "SUMMARY.json")
            if not p.exists():
                return "PENDING FULL RUN (security outputs absent)"
            s = json.loads(p.read_text())
            return f"OK ({s.get('mode')}; Analysis A appendix-only, D38)"
        if name == "00b_external_proxies":
            ext = CFG_REPO["external"]
            p = (out_path(ext["proxy_file"]) if MODE == "full"
                 else out_path(ext["out_subdir"], MODE, ext["proxy_file"]))
            if not p.exists():
                return "PENDING FULL RUN (external proxy absent)"
            return "OK (FULL)" if MODE == "full" else "OK (sample)"
        if name == "07_security_models":
            # full/synthetic runs write SUMMARY.json; only the sample placeholder writes the CSV
            p = out_path("security_models", "SUMMARY.json")
            summary_obj = None
            if p.exists():
                summary_obj = json.loads(p.read_text())
                if "B" in (summary_obj.get("failed_blocks") or {}):
                    return "FAILED (B block)"
            p = out_path("security_models", "b_primary.csv")
            if p.exists():
                try:
                    _first = pd.read_csv(p, nrows=1)
                    if ("block_status" in _first.columns and len(_first)
                            and str(_first["block_status"].iloc[0]).strip().upper() == "FAILED"):
                        return "FAILED (B block)"
                except Exception:
                    pass
            if summary_obj is not None:
                prov = str(summary_obj.get("provenance", ""))
                return "OK" if prov == "FULL" else "OK (synthetic)"
            p = out_path("security_models", "security_models_summary.csv")
            if p.exists():
                st = str(pd.read_csv(p)["status"].iloc[0])
                return st if st == "PENDING FULL RUN" else "OK"
            return "PENDING FULL RUN"
        if name == "10_dose_response":
            ddir = out_path("security_models", str(CFG_REPO["dose"]["out_subdir"]))
            sj = ddir / "SUMMARY.json"
            if sj.exists():
                prov_d = str(json.loads(sj.read_text()).get("provenance", ""))
                return "OK" if prov_d == "FULL" else "OK (synthetic)"
            if (ddir / "dose_summary.csv").exists():
                return str(pd.read_csv(ddir / "dose_summary.csv")["status"].iloc[0])
            return "PENDING FULL RUN"
        sub, fn = SUMMARY_FILE[name]
        p = out_path(sub, fn)
        if p.exists():
            st = str(pd.read_csv(p)["status"].iloc[0])
            return st if st == "PENDING FULL RUN" else "OK"
    except Exception:
        pass
    return "OK"

# `--force` is per-stage: FORCE=True forces every stage; otherwise only FORCE_STAGES is forced.
def force_args(stage_name):
    return ["--force"] if (FORCE or stage_name in FORCE_STAGES) else []

_unknown_force = [s for s in FORCE_STAGES if s not in STAGE_ORDER]
if _unknown_force:
    print(f"WARNING: FORCE_STAGES has unknown stage(s): {_unknown_force}")

# RUN_ONLY (D45): a single-stage re-run is a supported path, not a hand-edit of STAGE_ORDER.
RUN_ONLY_NORM = [str(s).strip() for s in (RUN_ONLY or []) if str(s).strip()]
_unknown_only = [s for s in RUN_ONLY_NORM if s not in STAGE_ORDER]
if _unknown_only:
    print(f"WARNING: RUN_ONLY has unknown stage(s), ignored: {_unknown_only}")
STAGES_TO_RUN = [s for s in STAGE_ORDER if (not RUN_ONLY_NORM or s in RUN_ONLY_NORM)]
SKIPPED_STAGES = [s for s in STAGE_ORDER if s not in STAGES_TO_RUN]
if RUN_ONLY_NORM and not STAGES_TO_RUN:
    raise ValueError(f"RUN_ONLY={RUN_ONLY_NORM} matches no stage in STAGE_ORDER")

RUN_RESULTS = []

# ---- D49 checkpoint fingerprints: say WHY a checkpoint was rejected -----------
_STAGE_MARKERS = {
    "00_thresholds": ("", "thresholds.json"),
    "02_combine_envelope": ("envelope", "_DONE"),
    "03_game_demand": ("game", "_DONE"),
    "04_models": ("models", "_DONE"),
    "05_validate": ("validate", "_DONE"),
    "06_security": ("security", "_DONE.json"),
    "07_security_models": ("security_models", "_DONE.json"),
    "10_dose_response": ("security_models/dose", "_DONE.json"),
}

def checkpoint_fingerprint_state(name):
    """(old, new) when an existing marker's config fingerprint changed, else None."""
    if name in LC_STAGES:
        # LC checkpoints: data/interim/checkpoints/<stage>__<mode>__<fingerprint16>.json (D49 rule,
        # LC engine). Any marker for this stage+mode under a different fingerprint is stale.
        cur = LC.config_fingerprint(RUN_CFG, name)
        stale = []
        for p in sorted(LC_CKPT.glob(f"{name}__{MODE}__*.json")):
            old = (read_json(p) or {}).get("config_fingerprint")
            if old and old != cur:
                stale.append(old)
        return (stale[0], cur) if stale else None
    spec = _STAGE_MARKERS.get(name)
    if spec is None:
        return None
    marker = out_path(*[p for p in spec if p])
    rec = common.read_done_marker(marker)
    if not rec:
        return None
    new = common.config_fingerprint(RUN_CFG, name)
    old = rec.get("config_fingerprint")
    return None if old == new else (old, new)
if SKIPPED_STAGES and not CHECKOUT_BLOCKED:
    print(f"RUN_ONLY={RUN_ONLY_NORM} — running {len(STAGES_TO_RUN)}/{len(STAGE_ORDER)} stages; "
          "the rest are SKIPPED and their checkpoints are reused:")
    for name in SKIPPED_STAGES:
        print(f"  [{'SKIPPED - checkpoint':<44}] {name:<20} "
              f"{run_diagnostics.stage_status_on_disk(name, OUTPUTS)}")
    print()
if CHECKOUT_BLOCKED:
    print("pipeline SKIPPED — checkout guard blocked this run")
    print(f"  {CHECKOUT['stop_message']}")
    print("  No stage was imported or executed. Pull the checkout and re-run this cell.")
    RUN_RESULTS = []
else:
    print(f"running pipeline in '{MODE}' mode — every stage reloads its code from disk (D46) ...\n")
    STALE_FINGERPRINT = []
    for name in STAGES_TO_RUN:
        _fp_state = checkpoint_fingerprint_state(name)
        res = run_stage(name, ["--mode", MODE, *force_args(name)])
        if res["ok"]:
            res["status"] = stage_output_status(name)
        else:
            res["status"] = "PENDING FULL RUN" if name in GAME_SIDE else "FAILED"
        RUN_RESULTS.append(res)
        code_note = ("code reloaded from disk" if res.get("module_action") == "reloaded"
                     else "code imported from disk")
        print(f"[{res['status']:<44}] {name:<20} {res['seconds']:>7.2f}s  {code_note}")
        if not res["ok"]:
            for line in run_diagnostics.failure_lines(res):
                print(f"           {line}")
        if _fp_state is not None:
            _old, _new = _fp_state
            STALE_FINGERPRINT.append(name)
            print("           checkpoint REJECTED — config fingerprint changed "
                  f"({'missing legacy marker' if _old is None else str(_old)[:12]} -> "
                  f"{str(_new)[:12]}); rebuilding from the current config")

# D49: a changed/missing fingerprint is not a normal resume. Say it once, plainly.
if not CHECKOUT_BLOCKED and STALE_FINGERPRINT:
    print("!" * 88)
    print("artefacts stale — ONE FULL REBUILD REQUIRED")
    print(f"  {len(STALE_FINGERPRINT)} checkpoint(s) were written under a different config "
          "fingerprint (D49): " + ", ".join(STALE_FINGERPRINT))
    print("  Those stages rebuilt from the current config above. Any stage skipped by RUN_ONLY")
    print("  must also be rebuilt before its numbers are quoted — the old artefacts are stale.")
    print("!" * 88)


## 4. Synthetic-fixture demonstration (game-side path)

The Pi does not hold the full game-tracking files, so the game-side stages emit
`PENDING FULL RUN` placeholders above. To show that path **actually runs**, this cell drives
the committed, schema-compliant fixture (`fixtures/synthetic/`) with a **planted binding
ceiling** and a **null** control. Fixture outputs are labelled *not a finding* — they prove
the plumbing, nothing about the NFL. `06_security` writes the same folder in synthetic and real
modes, so the cell finishes by re-running it in `MODE` to restore the real (mode-labelled) output.

In [ ]:
SYNTH_RESULTS = []
if CHECKOUT_BLOCKED:
    print("synthetic demo skipped — checkout guard blocked")
elif RUN_SYNTHETIC_DEMO:
    for regime in SYNTHETIC_REGIMES:
        print(f"--- synthetic regime: {regime} ---")
        # D63: 00_thresholds is now the LC stage and has no --synthetic mode; the ceiling-v1
        # synthetic path starts at 03_game_demand (it reads the committed fixture directly).
        for name in ("03_game_demand", "00b_external_proxies", "04_models", "05_validate"):
            args = (["--synthetic", *force_args(name)] if name == "00b_external_proxies"
                    else ["--synthetic", "--synthetic-regime", regime, *force_args(name)])
            res = run_stage(name, args)
            res["status"] = "OK (synthetic)" if res["ok"] else "FAILED"
            SYNTH_RESULTS.append(res)
            why = "" if res["ok"] else (res["error"] or (res["tail"][-1] if res["tail"] else "failed"))
            print(f"  [{res['status']:<14}] {name:<20} {res['seconds']:>7.2f}s" + (f"  {why}" if why else ""))
    # security: the pre-Combine exposure/Analysis-A stage + the B–G incentive-exposure stage
    for regime in ("planted", "null"):
        res = run_stage("06_security", ["--synthetic", "--synthetic-regime", regime, *force_args("06_security")])
        res["status"] = "OK (synthetic)" if res["ok"] else "FAILED"
        SYNTH_RESULTS.append(res)
        print(f"  [{res['status']:<14}] {'06_security':<20} {res['seconds']:>7.2f}s"
              + ("" if res["ok"] else f"  {res['error']}"))
    for regime in SECURITY_REGIMES:
        res = run_stage("07_security_models", ["--synthetic", "--synthetic-regime", regime, *force_args("07_security_models")])
        res["status"] = "OK (synthetic)" if res["ok"] else "FAILED"
        SYNTH_RESULTS.append(res)
        print(f"  [{res['status']:<14}] {'07_security_models':<20} {res['seconds']:>7.2f}s"
              + ("" if res["ok"] else f"  {res['error']}"))
    # 06_security writes outputs/security/ in both real and synthetic modes. The demo above
    # would otherwise leave SYNTHETIC files in the real output folder, so restore the real run.
    res = run_stage("06_security", ["--mode", MODE, *force_args("06_security")])
    print(f"  [{'OK' if res['ok'] else 'FAILED':<14}] {'06_security':<20} {res['seconds']:>7.2f}s"
          f"  (restored {MODE}-mode outputs)")
else:
    print("synthetic demo skipped (RUN_SYNTHETIC_DEMO=False)")


## 5. Security / incentive exposure (D38)

Reads `outputs/security_models/` (real if present, else the synthetic `planted` / `null` /
`selection` fixture). The tables here are **descriptive display**; the single pre-declared WR `moved_down`
coefficient and the formal verdict are in §8. A **positive** `group_coef` = `moved_down` has a **smaller**
Combine→game gap, i.e. the incentive hypothesis; a **negative** one is evidence **against** it.

**Group assignment is pre-Combine only.** D40 asymmetric threshold: `moved_down` iff
`edp_trend >= +3.0` — the player fell **3 or more EDP spots** between the 2-week and 1-week
pre-Combine snapshots. `high_secure` is **everyone else with a trend** (`edp_trend < +3.0`):
**stable or rising**, and deliberately not a "low EDP" group. A +5 cut is a declared sensitivity
(`panel_threshold_T5`). Absent a 2-week snapshot, within-position tertiles of `log(EDP_pre)` are
used and the middle tertile is `unclassified`. Post-Combine EDP is never used.

If the real `07_security_models` output is absent the reader falls back to the committed fixture and
says so; with `SHOW_SYNTHETIC_PREVIEW = False` (default) the synthetic tables are **withheld**
rather than shown as numbers.

In [ ]:
SEC_MODELS = out_path("security_models")

# The real security chain, in order: 06_security writes outputs/security/ (board + Combine only),
# then 07_security_models writes outputs/security_models/ (B-G). Each step names its own absence,
# so a fallback is never silent (D41). Crashes are reported as crashes: a missing file after a
# stage exception is NEVER blamed on missing data (D43 hardening).
def _last_run_result(name):
    results = globals().get("RUN_RESULTS") or []
    return next((x for x in reversed(results) if x.get("stage") == name), None)

def _first_row(path):
    try:
        df = pd.read_csv(path, nrows=1)
        return df.iloc[0].to_dict() if len(df) else None
    except Exception:
        return None

def security_chain_issues():
    """(reasons, actions) - why the REAL security chain is not quotable, in pipeline order."""
    reasons, actions = [], []

    p6 = out_path("security", "SUMMARY.json")
    if not p6.exists():
        r6 = _last_run_result("06_security")
        if r6 is not None and r6.get("error"):
            reasons.append(f"06_security crashed: {r6['error']}")
            reasons.extend(run_diagnostics.failure_lines(r6)[:12])
            actions.append("fix the 06_security exception and re-run it "
                           "(the board file is not the problem)")
        else:
            tail = "" if (r6 is None or r6.get("ok")) else f" (stage exit {r6.get('rc')}; see §3 log)"
            reasons.append("outputs/security/SUMMARY.json absent → stage 06_security did not run" + tail)
            if r6 is not None and not r6.get("ok"):
                reasons.extend(run_diagnostics.failure_lines(r6)[:12])
                actions.append("resolve the 06_security stage blocker shown above")
            else:
                actions.append("supply the EDP board file at data/raw/external/gtm/ and re-run 06_security")
    else:
        try:
            p6_prov = str(json.loads(p6.read_text(encoding="utf-8")).get("provenance", "?"))
        except Exception as exc:
            p6_prov = f"unreadable ({type(exc).__name__})"
        if p6_prov != "FULL":
            reasons.append(f"outputs/security/SUMMARY.json provenance={p6_prov} "
                           "→ 06_security output is not a FULL run")

    p7 = SEC_MODELS / "b_primary.csv"
    full_game = bool(globals().get("full_game_ok", False))
    r7 = _last_run_result("07_security_models")
    row7 = _first_row(p7) if p7.exists() else None
    if not p7.exists():
        diag = run_diagnostics.diagnose_missing_primary(r7, full_game)
        reasons.append(f"{diag['status']}: {diag['reason']}")
        reasons.extend(diag["tail_lines"][:12])
        actions.append(diag["action"])
    elif row7 is not None and str(row7.get("block_status", "")).upper() == "FAILED":
        diag = run_diagnostics.diagnose_missing_primary(r7, full_game, b_primary_row=row7)
        reasons.append(f"FAILED (B block): {diag['reason']}")
        actions.append(diag["action"])
    else:
        try:
            p7_prov = str(pd.read_csv(p7, usecols=["provenance"])["provenance"].iloc[0])
        except Exception:
            p7_prov = "missing"
        if p7_prov != "FULL":
            reasons.append(f"outputs/security_models/b_primary.csv provenance={p7_prov}, "
                           "not FULL — rebuild in full mode")
            actions.append("run 07_security_models in MODE='full' with FULL thresholds and rebuild")
    return reasons, actions


def security_dir():
    """(dir | None, is_synthetic | None, reason) - where §5/§8 read the security models, and why.

    Real 07 outputs win, but a b_primary.csv whose first row is block_status=FAILED is not a
    result: it is a labelled failure. In that case we fall through to the same honest absence
    logic (and never silently substitute the fixture for a number). D41/D43.
    """
    real = SEC_MODELS / "b_primary.csv"
    if real.exists():
        row = _first_row(real)
        status = str((row or {}).get("block_status", "")).upper()
        if status != "FAILED":
            return SEC_MODELS, False, f"real output: {rel(real)}"
    reasons, _ = security_chain_issues()
    because = "; ".join(reasons) if reasons else "no real 07 output"
    for reg in ("planted", "selection", "null"):
        d = SEC_MODELS / "synthetic" / reg
        if (d / "b_primary.csv").exists():
            return d, True, f"synthetic fixture '{reg}' (NOT a finding) because {because}"
    return None, None, because

SEC_DIR, SEC_SYNTH, SEC_WHY = security_dir()
if SEC_DIR is None:
    print("no security-model outputs yet — run 07_security_models (or the synthetic demo)")
    print(f"  why: {SEC_WHY}")
else:
    print(f"source    : {rel(SEC_DIR)}")
    print(f"data kind : {'SYNTHETIC FIXTURE — NOT A FINDING' if SEC_SYNTH else 'real stage output'}")
    print(f"why       : {SEC_WHY}")
    if SEC_SYNTH and not SHOW_SYNTHETIC_PREVIEW:
        print("\nSYNTHETIC NUMBERS WITHHELD — the tables below would show the tuned fixture, not data.")
        print("  Set SHOW_SYNTHETIC_PREVIEW=True in the Configuration cell to render the layout preview.")
    else:
        if SEC_SYNTH:
            print("\nSYNTHETIC LAYOUT PREVIEW — NOT DATA (SHOW_SYNTHETIC_PREVIEW=True).")
        sec_b = read_csv(SEC_DIR / "b_primary.csv")
        sec_c = read_csv(SEC_DIR / "c_end.csv")
        sec_x = read_csv(SEC_DIR / "c_exceedance.csv")
        sec_assign = read_csv(SEC_DIR / "group_assignment.csv")
        sec_sum = json.loads((SEC_DIR / "SUMMARY.json").read_text()) if (SEC_DIR / "SUMMARY.json").exists() else {}
        sec_prov = prov_of(sec_b)
        print(f"source: {SEC_DIR.relative_to(REPO)}   provenance: {sec_prov}")
        print("Analysis A: retired as a headline gate by D38 — appendix only.")
        print("Sign: a POSITIVE group_coef = moved_down has a SMALLER Combine→game gap (the hypothesis).")

        if len(sec_assign):
            st = sec_assign.groupby(["stratum", "group"]).size().reset_index(name="n")
            print("\nGroup counts (pre-Combine sort key):")
            display(st)
        if len(sec_b):
            b = sec_b[sec_b["spec"] == "primary_env_vmax"][
                ["scope", "n_obs", "n_players", "group_coef", "group_se", "group_p",
                 "group_ci_lo", "group_ci_hi", "group_coef_corrected"]].copy()
            for c in ("group_coef", "group_se", "group_p", "group_ci_lo", "group_ci_hi",
                      "group_coef_corrected"):
                b[c] = to_num(b[c])
            print("\nB — primary effort-level group coefficient (player-clustered SEs):")
            display(b)
        if len(sec_x):
            print("\nC — exceedance (overall first, then by group), WR:")
            display(sec_x[sec_x["scope"] == "WR"][
                ["group", "n_efforts", "n_exceed", "exceedance_rate", "wilson_lo", "wilson_hi"]])
        if len(sec_c):
            c = sec_c[(sec_c["scope"] == "WR") & (sec_c["basis"] == "mean_ceiling")][
                ["percentile", "log_edp_pre", "fitted_gap", "se", "ci_lo", "ci_hi"]].copy()
            for col in ("percentile", "log_edp_pre", "fitted_gap", "se", "ci_lo", "ci_hi"):
                c[col] = to_num(c[col])
            print("\nC — fitted gap by log(EDP_pre) percentile (WR, mean ceiling):")
            display(c)
        if SEC_SYNTH:
            print("\nSYNTHETIC FIXTURE (not a finding) — these numbers prove the plumbing only.")


## 6. Writeup figures (≤5)

Figures are rendered from the pipeline outputs into `outputs/figures/` and shown inline.
Pre-full-run, **Figure 1** runs on the complete Combine file but is stamped `UNVALIDATED SAMPLE OUTPUT`
(the label follows `MODE`, not the file's completeness); the game-side figures fall back to the
synthetic fixture, carrying its *not a finding* label. When `MODE="full"` has produced
real outputs, the same code picks those up automatically and labels them `FULL`.

**Figure slide (D38):** the old *feature-selection ICC/VIF* bar chart is **retired** — its data
remain in `envelope/feature_selection.csv` and are reported as a Methods table. The slot now carries
the **security / incentive-exposure** figure, so the committed set stays at **5**.

### Visualisation briefs

| # | chart_type | data_source | columns_required | output_filename |
|---|---|---|---|---|
| 1 | line + band | `envelope/per_player_envelope.csv` | `combine_position, env_vmax, env_tau, peak_speed` | `fig1_combine_envelope_decay.png` |
| 2 | 3-panel (counts / coef / gap) | `security_models/{group_assignment,b_primary,c_end}.csv` | `stratum, group, scope, group_coef, group_ci_lo, group_ci_hi, percentile, fitted_gap` | `fig2_security_incentive_exposure.png` |
| 3 | grouped bar | `models/*/exceedance_utilization.csv` | `pooling, exceedance_rate, u_median` | `fig3_exceedance_utilization.png` |
| 4 | line + error bars | `models/*/model_coefficients.csv` | `pooling, tau, reported, ci_lo, ci_hi, unstable` | `fig4_quantile_slope_vs_tau.png` |
| 5 | multi-panel | `validate/*/{cv_results,measurement_error,permutation}.csv` | `tau, model, pinball, slope_uncorrected, slope_corrected, p_two_sided` | `fig5_validation_cv_permutation.png` |

*Retired:* `fig2_feature_selection_icc_vif.png` (now a Methods table, not a figure).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# ---- figure helpers (output readers are defined in the bootstrap cell) ------
def save_and_show(fig, filename):
    path = FIGDIR / filename
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    if SHOW_FIGURES:
        plt.show()
    print(f"saved -> {path.relative_to(REPO)}   ({path.stat().st_size} bytes)")
    return path

def models_series():
    """(label, dir, is_synthetic, reason) for 04_models outputs: real if present, else the fixture.

    The fallback is never silent: `reason` names the missing real output (D41).
    """
    real = out_path("models", "model_coefficients.csv")
    if real.exists():
        return [("FULL", out_path("models"), False, f"real output: {rel(real)}")]
    why = ("outputs/models/model_coefficients.csv absent → stage 04_models produced no estimates "
           "(sample mode writes a PENDING placeholder; the game side needs the full tracking files)")
    out = []
    for reg in ("binding", "null"):
        d = out_path("models", "synthetic", reg)
        if (d / "model_coefficients.csv").exists():
            out.append((f"synthetic:{reg}", d, True, why))
    return out

def validate_series():
    """(label, dir, is_synthetic, reason) for 05_validate outputs: real if present, else the fixture."""
    real = out_path("validate", "cv_results.csv")
    if real.exists():
        return [("FULL", out_path("validate"), False, f"real output: {rel(real)}")]
    why = ("outputs/validate/cv_results.csv absent → stage 05_validate produced no estimates "
           "(sample mode writes a PENDING placeholder; the game side needs the full tracking files)")
    out = []
    for reg in ("binding", "null"):
        d = out_path("validate", "synthetic", reg)
        if (d / "cv_results.csv").exists():
            out.append((f"synthetic:{reg}", d, True, why))
    return out

def announce_series(series):
    """Print where each plotted series came from — a fixture fallback is stated, never implied."""
    for label, d, is_synth, why in series:
        print(f"source [{label}]: {rel(d)}"
              + (f"  — SYNTHETIC FIXTURE (not a finding): {why}" if is_synth else ""))

FIGURES = []

### Figure 1 — Combine ceiling envelope (`v(t)=env_vmax·(1−e^{−t/env_tau})`)

In [ ]:
env = read_csv(out_path("envelope", "per_player_envelope.csv"))
if not len(env):
    print("envelope outputs missing — run 02_combine_envelope first")
else:
    prov1 = prov_of(env)
    t = np.linspace(0.0, 3.0, 200)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
    for pos, col in (("WR", "C0"), ("TE", "C1")):
        sub = env[env["combine_position"] == pos]
        vm, tau = to_num(sub["env_vmax"]).to_numpy(), to_num(sub["env_tau"]).to_numpy()
        m = np.isfinite(vm) & np.isfinite(tau) & (tau > 0)
        if not m.any():
            continue
        curves = vm[m, None] * (1.0 - np.exp(-t[None, :] / tau[m, None]))
        a1.plot(t, np.median(curves, 0), color=col, label=f"{pos} median (n={m.sum()})")
        a1.fill_between(t, np.percentile(curves, 25, 0), np.percentile(curves, 75, 0),
                        color=col, alpha=0.18)
        a2.scatter(to_num(sub["peak_speed"]), to_num(sub["env_vmax"]),
                   s=14, alpha=0.6, color=col, label=pos)
    a1.set_xlabel("time from movement onset (s)"); a1.set_ylabel("speed (yd/s)")
    a1.set_title("Fitted ceiling envelope v(t)"); a1.legend(fontsize=8)
    lim = list(a2.get_xlim()) + list(a2.get_ylim())
    lo, hi = min(lim), max(lim)
    a2.plot([lo, hi], [lo, hi], "k--", lw=1, label="y=x")
    a2.set_xlabel("model-free peak speed (yd/s)"); a2.set_ylabel("fitted ceiling env_vmax (yd/s)")
    a2.set_title("Fitted ceiling vs raw peak"); a2.legend(fontsize=8)
    fig.suptitle(f"Fig 1 — Combine ceiling envelope   [{prov1}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig1_combine_envelope_decay.png"))

### Figure 2 — Security / incentive exposure (group counts, B contrast, C dose-response)

In [ ]:
SEC_DIR, SEC_SYNTH, SEC_WHY = security_dir()
if SEC_DIR is None:
    print("no security-model outputs yet (run 07_security_models, or the synthetic demo)")
else:
    print(f"fig2 source: {rel(SEC_DIR)}"
          + ("  — SYNTHETIC FIXTURE (not a finding): " + SEC_WHY if SEC_SYNTH else ""))
    assign = read_csv(SEC_DIR / "group_assignment.csv")
    b = read_csv(SEC_DIR / "b_primary.csv")
    c = read_csv(SEC_DIR / "c_end.csv")
    prov2 = prov_of(b)

    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14, 4.3))

    # (1) group counts by stratum
    if len(assign):
        g = assign.groupby(["stratum", "group"]).size().unstack(fill_value=0)
        idx = np.arange(len(g.index)); w = 0.8 / max(len(g.columns), 1)
        for j, col in enumerate(g.columns):
            a1.bar(idx + (j - (len(g.columns) - 1) / 2) * w, g[col], width=w, label=col)
        a1.set_xticks(idx); a1.set_xticklabels(list(g.index), rotation=15)
        a1.set_ylabel("players"); a1.legend(fontsize=8)
    a1.set_title("pre-Combine sort key")

    # (2) B group coefficient by scope (primary spec)
    if len(b):
        bb = b[b["spec"] == "primary_env_vmax"].copy()
        bb["_c"] = to_num(bb["group_coef"]); bb["_lo"] = to_num(bb["group_ci_lo"])
        bb["_hi"] = to_num(bb["group_ci_hi"])
        xs = np.arange(len(bb))
        a2.errorbar(xs, bb["_c"], yerr=[bb["_c"] - bb["_lo"], bb["_hi"] - bb["_c"]],
                    fmt="o", capsize=4, color="C0")
        a2.axhline(0, ls="--", c="k", lw=1)
        a2.set_xticks(xs); a2.set_xticklabels(bb["scope"])
        a2.set_ylabel("group coefficient (log speed)")
    a2.set_title("B: moved_down vs high_secure")

    # (3) C fitted gap by log(EDP) percentile
    if len(c):
        cc = c[(c["scope"] == "WR") & (c["basis"] == "mean_ceiling")].sort_values("percentile")
        if len(cc):
            x = to_num(cc["percentile"]); yv = to_num(cc["fitted_gap"])
            lo, hi = to_num(cc["ci_lo"]), to_num(cc["ci_hi"])
            a3.errorbar(x, yv, yerr=[yv - lo, hi - yv], fmt="-o", capsize=4)
        a3.set_xlabel("log(EDP_pre) percentile"); a3.set_ylabel("fitted gap (log speed)")
    a3.set_title("C: which end drives it (WR)")

    fig.suptitle(f"Fig 2 — Security / incentive exposure   [{prov2}]")
    fig.tight_layout()
    FIGURES.append(save_and_show(fig, "fig2_security_incentive_exposure.png"))
    print("A positive group coefficient = moved_down has the SMALLER gap (the hypothesis).")
    if SEC_SYNTH:
        print("SYNTHETIC FIXTURE (not a finding).")

### Figure 3 — Exceedance & utilization (game side)

In [ ]:
series = models_series()
if not series:
    print("no 04_models outputs yet (run 04_models, or the synthetic demo)")
else:
    announce_series(series)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
    pools = ["pooled", "WR", "TE"]
    x = np.arange(len(pools)); w = 0.8 / max(len(series), 1)
    synth = False
    for i, (label, d, is_synth, why) in enumerate(series):
        synth = synth or is_synth
        ex = read_csv(Path(d) / "exceedance_utilization.csv")
        if not len(ex):
            continue
        ex = ex.set_index("pooling")
        xs = x + (i - (len(series) - 1) / 2) * w
        a1.bar(xs, [to_num(ex["exceedance_rate"]).get(p, np.nan) for p in pools],
               width=w, label=label)
        a2.bar(xs, [to_num(ex["u_median"]).get(p, np.nan) for p in pools], width=w, label=label)
    a1.axhline(0.25, ls="--", c="r", lw=1); a1.text(0.01, 0.26, "reframe >0.25",
                                                    color="r", fontsize=7, transform=a1.get_yaxis_transform())
    a1.set_xticks(x); a1.set_xticklabels(pools); a1.set_ylabel("exceedance rate")
    a1.set_title("Share of efforts beating the ceiling"); a1.legend(fontsize=8)
    a2.axhline(1.0, ls="--", c="k", lw=1); a2.set_xticks(x); a2.set_xticklabels(pools)
    a2.set_ylabel("median utilization u = effort / ceiling")
    a2.set_title("Utilization (1.0 = at ceiling)"); a2.legend(fontsize=8)
    tag = "SYNTHETIC FIXTURE (not a finding)" if synth else "FULL"
    fig.suptitle(f"Fig 3 — Exceedance & utilization   [{tag}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig3_exceedance_utilization.png"))

### Figure 4 — Quantile slope vs τ (does the link strengthen toward the tail?)

In [ ]:
series = models_series()
if not series:
    print("no 04_models outputs yet (run 04_models, or the synthetic demo)")
else:
    announce_series(series)
    fig, ax = plt.subplots(figsize=(8.5, 4.6))
    styles = {"WR": "-o", "TE": "--s"}
    synth = False
    for label, d, is_synth, why in series:
        synth = synth or is_synth
        mc = read_csv(Path(d) / "model_coefficients.csv")
        if not len(mc):
            continue
        mc = mc[(mc["feature"] == "env_vmax") & (mc["controls"] == "without_draft")]
        for pool, ls in styles.items():
            sub = mc[mc["pooling"] == pool].sort_values("tau")
            if not len(sub):
                continue
            est = to_num(sub["reported"])          # blank (null) reported -> NaN -> not plotted
            lo, hi = to_num(sub["ci_lo"]), to_num(sub["ci_hi"])
            ax.errorbar(to_num(sub["tau"]), est, yerr=[est - lo, hi - est],
                        fmt=ls, capsize=3, alpha=0.9, label=f"{label} · {pool}")
            un = is_true(sub["unstable"]) & est.notna()
            ax.scatter(to_num(sub["tau"])[un], est[un], marker="x", s=45, color="k", zorder=5)
    ax.axhline(0.0, lw=1, color="k", alpha=0.5)
    ax.set_xlabel("τ (quantile)"); ax.set_ylabel("slope on ceiling env_vmax  (in-game speed per yd/s)")
    ax.set_title("Quantile slope vs τ  (×  = bootstrap-unstable; blanks = reported null)")
    ax.legend(fontsize=8)
    tag = "SYNTHETIC FIXTURE (not a finding)" if synth else "FULL"
    fig.suptitle(f"Fig 4 — Ceiling → game link across the tail   [{tag}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig4_quantile_slope_vs_tau.png"))

### Figure 5 — Validation: player-grouped CV vs controls-only, measurement error, permutation

In [ ]:
series = validate_series()
if not series:
    print("no 05_validate outputs yet (run 05_validate, or the synthetic demo)")
else:
    announce_series(series)
    label, d, is_synth, why = series[0]         # binding (or real) is the reference
    cv = read_csv(Path(d) / "cv_results.csv")
    me = read_csv(Path(d) / "measurement_error.csv")
    pm = read_csv(Path(d) / "permutation.csv")
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14, 4.2))

    if len(cv):
        g = cv.groupby(["tau", "model"])["pinball"].apply(lambda s: to_num(s).mean()).unstack()
        taus = list(g.index)
        x = np.arange(len(taus)); w = 0.38
        if "full" in g:      a1.bar(x - w/2, g["full"],          width=w, label="full")
        if "controls_only" in g: a1.bar(x + w/2, g["controls_only"], width=w, label="controls-only baseline")
        a1.set_xticks(x); a1.set_xticklabels([str(t) for t in taus])
    a1.set_xlabel("τ"); a1.set_ylabel("mean pinball loss (lower = better)")
    a1.set_title("Player-grouped CV vs baseline"); a1.legend(fontsize=8)

    if len(me):
        mt = to_num(me["tau"])
        a2.plot(mt, to_num(me["slope_uncorrected"]), "-o", label="uncorrected")
        a2.plot(mt, to_num(me["slope_corrected"]), "--s", label="ICC-attenuation corrected")
        if len(me) and "reliability_icc" in me:
            a2.set_title(f"Measurement error\n(reliability ICC ≈ {to_num(me['reliability_icc']).mean():.3f})",
                         fontsize=10)
    a2.axhline(0.0, lw=1, color="k", alpha=0.5)
    a2.set_xlabel("τ"); a2.set_ylabel("slope"); a2.legend(fontsize=8)

    if len(pm):
        r = pm.iloc[0]
        obs = abs(float(to_num(pd.Series([r["observed_slope"]]).iloc[0])))
        p = float(to_num(pd.Series([r["p_two_sided"]]).iloc[0]))
        a3.bar(["observed\n|slope|"], [obs], color="C3", width=0.5)
        a3.axhline(0.0, lw=1, color="k", alpha=0.5)
        a3.set_title(f"Permutation null (τ={r['tau']})\np = {p:.3f}  "
                     f"({int(r['n_ge'])}/{int(r['n_reps'])} perms ≥ obs)", fontsize=9)
        a3.set_ylabel("|slope|")
    tag = "SYNTHETIC FIXTURE (not a finding)" if is_synth else "FULL"
    fig.suptitle(f"Fig 5 — Validation   [{tag}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig5_validation_cv_permutation.png"))

## 7. Run summary

A consolidated status for the whole run, the per-stage wall-time / peak-RAM table, and the
`PENDING FULL RUN` / `PARTIAL` / `FAILED` items. In `sample` mode every number is stamped `UNVALIDATED SAMPLE OUTPUT`;
`full` mode is the real analysis; synthetic fixtures are `SYNTHETIC FIXTURE (not a finding)`.

In [ ]:
def stage_status(res):
    return res.get("status") or ("OK" if res["ok"] else "FAILED")

summary = pd.DataFrame([{"stage": r["stage"], "status": stage_status(r),
                         "seconds": r["seconds"], "error": r["error"]} for r in RUN_RESULTS])
if SYNTH_RESULTS:
    summary = pd.concat([summary, pd.DataFrame(
        [{"stage": f"{r['stage']} [synthetic]", "status": stage_status(r),
          "seconds": r["seconds"], "error": r["error"]} for r in SYNTH_RESULTS])],
        ignore_index=True)

prov_file = out_path("thresholds.json")
prov = json.loads(prov_file.read_text())["provenance"] if prov_file.exists() else "n/a"
pending = summary.loc[summary["status"].str.startswith("PENDING"), "stage"].tolist()
partial = summary.loc[summary["status"].str.startswith("PARTIAL"), "stage"].tolist()
failed  = summary.loc[summary["status"].str.startswith("FAILED"), "stage"].tolist()

print(f"MODE = {MODE}   ·   thresholds provenance = {prov}")
print(f"label            : " + ("UNVALIDATED SAMPLE OUTPUT" if MODE != "full"
                               else "FULL (real analysis)"))
print(f"figures written  : {len(FIGURES)} of max 5  ->  {FIGDIR.relative_to(REPO)}/"
      "   (retired: fig2_feature_selection_icc_vif.png; replaced by fig2_security_incentive_exposure.png)")
print(f"PENDING FULL RUN : {pending if pending else 'none'}")
print(f"PARTIAL          : {partial if partial else 'none'}")
print(f"FAILED           : {failed if failed else 'none'}")
if globals().get("SKIPPED_STAGES"):
    print(f"SKIPPED (RUN_ONLY, checkpoint reused) : {SKIPPED_STAGES}")
display(summary)

# ---- runtime / peak-RAM table (from outputs/run_log.csv) --------------------
rl = out_path("run_log.csv")
if rl.exists():
    log = read_csv(rl)
    # D63: one run log can hold both engines' rows — the LC engine names the RAM column
    # `peak_rss_mb`, ceiling-v1 names it `peak_ram_mb`. Read whichever is present; never assume.
    ram_col = next((c for c in ("peak_rss_mb", "peak_ram_mb") if c in log.columns), None)
    if "wall_s" not in log.columns or not len(log):
        print("\nrun log present but has no wall_s column — no timing table")
    else:
        log["_s"] = to_num(log["wall_s"])
        log["_m"] = to_num(log[ram_col]) if ram_col else float("nan")
        rt = (log[log["_s"] > 0].groupby("stage")
              .agg(runs=("_s", "size"), median_wall_s=("_s", "median"),
                   max_wall_s=("_s", "max"), peak_ram_mb=("_m", "max"))
              .reset_index().sort_values("stage"))
        print(f"\nwall time / peak RAM per stage (outputs/run_log.csv; RAM column: {ram_col or 'absent'}):")
        display(rt)

## 8. Headline results — read this first

This section reads the output files and prints the **pre-declared** verdict in plain English.
Nothing here is typed in by hand: every number is loaded from `outputs/security_models/`.

**How to read the label on this section**

* **`FULL RUN`** — a real full-data run with `FULL` thresholds and `FULL` security outputs. The
  verdict is live.
* **`NO RESULT — synthetic/sample data in use`** — `MODE` is not `full`, a stage is missing or
  failed, or the thresholds / security provenance is not `FULL`. The headline block is **not printed
  at all** — no coefficient, CI, p-value, effect size or utilisation mean — because the committed
  fixture is tuned to look plausible and would read as a finding. The section names the exact
  missing input and the next step instead.
* **`SYNTHETIC LAYOUT PREVIEW — NOT DATA`** — only when `SHOW_SYNTHETIC_PREVIEW = True` (set it in the
  Configuration cell). It renders the fixture so the layout can be checked; it is never a result.
  Leave the flag `False` for anything Conor or a reader will see.

**The test (fixed before the full run)**

* **H1:** a player whose pre-Combine draft stock is slipping (`moved_down`) realises a *higher*
  share of his Combine ceiling in games — a *smaller* Combine→game gap — than a high/secure player.
* **Sign convention:** `group_coef` is the `moved_down` − `high_secure` contrast on log game speed.
  **Positive = `moved_down` is closer to its ceiling = supports H1.** Negative = against H1.
* **Groups (D40 asymmetric threshold):** `moved_down` = fell **3+ EDP spots** between the
  2-week and 1-week pre-Combine snapshots (`edp_trend >= +3.0`). `high_secure` = **not
  falling 3+ (stable or rising)** — not a "low EDP" group. The declared +5 sensitivity is
  reported as `panel_threshold_T5`.
* **Pre-declared verdict rule:**
  * `SUPPORTED` — `group_coef > 0`, `p < 0.05`, 95% CI lower bound > 0, **and** the
    ICC-attenuation-corrected coefficient > 0;
  * `AGAINST` — `group_coef < 0`, `p < 0.05`, 95% CI upper bound < 0;
  * `NO TEST (thin group)` — `thin` is true, or either group has fewer than 15 players;
  * otherwise `INCONCLUSIVE (CI includes zero)`.

A null is reported as a null, and a naive significant estimate alone is never called supported.

**Checkpoint reuse:** stages that write a `_DONE` file are skipped on re-run unless named in
`FORCE_STAGES` (or `FORCE=True` for every stage); `CLEAN=True` deletes outputs first (only when
`CLEAN_CONFIRM="DELETE"`), including the Combine envelope, and therefore forces a ceiling refit.

**When the full data arrives:** put `game_tracking_{2023,24,25}.csv` where `DATA_DIR` points, set
`MODE="full"`, then **Run All**. The pipeline runs
`01_audit → 02_combine_envelope → 06_security → 00_thresholds → 03_game_demand → 07_security_models
→ 00b_external_proxies → 04_models → 05_validate`; game-side stages stay `PENDING FULL RUN` until
the full files and `FULL`-provenance thresholds exist.

In [ ]:
# ===================== 8. Headline results — computed, never typed =====================
# Every number is read from the output files. Missing files/columns are reported, never guessed.
# In sample/synthetic mode the verdict is deliberately suppressed.

CHECKOUT_BLOCKED = bool(CHECKOUT.get("blocked"))
if not CHECKOUT["state"]["git_available"]:
    print("checkout : git unavailable — continuing without a stale-code check")
else:
    for line in CHECKOUT["header_lines"]:
        print(f"checkout : {line}")
if CHECKOUT_BLOCKED:
    print("!" * 88)
    print("NO RESULT — checkout guard blocked the run")
    print(f"  {CHECKOUT['stop_message']}")
    print("  No numeric claim is printed: the pipeline did not run against this checkout.")
    print("!" * 88)

SEC_DIR_H, SEC_SYNTH_H, SEC_WHY_H = ((None, None, "checkout guard blocked")
                                     if CHECKOUT_BLOCKED else security_dir())
_THR = OUTPUTS / "thresholds.json"
thr_prov_h = None
if _THR.exists():
    try:
        thr_prov_h = str(json.loads(_THR.read_text(encoding="utf-8")).get("provenance"))
    except Exception as exc:
        thr_prov_h = f"unreadable ({type(exc).__name__})"

def _load(name):
    if SEC_DIR_H is None:
        return None, "no security-model outputs (07_security_models has not run)"
    p = SEC_DIR_H / name
    if not p.exists():
        return None, f"{p.relative_to(REPO)} missing"
    try:
        return pd.read_csv(p), ""
    except Exception as exc:
        return None, f"could not read {p.relative_to(REPO)}: {type(exc).__name__}: {exc}"

b_df, b_err = _load("b_primary.csv")
c_df, c_err = _load("c_end.csv")
m_df, m_err = _load("b_group_means.csv")
f_df, f_err = _load("falsification.csv")
i_df, i_err = _load("e_ipw.csv")

if b_df is None:
    sec_prov_h = "not available"
elif "provenance" in b_df.columns and len(b_df):
    sec_prov_h = str(b_df["provenance"].iloc[0])
else:
    sec_prov_h = "missing provenance column"

_b_row_status = ""
if b_df is not None and "block_status" in b_df.columns and len(b_df):
    _b_row_status = str(b_df["block_status"].iloc[0]).strip().upper()
# The FULL gate is provenance-driven and deliberately independent of RUN_RESULTS: a
# RUN_ONLY=["07_security_models"] re-run must still print a live headline block (D45).
FULL = run_diagnostics.headline_is_full(MODE, thr_prov_h, sec_prov_h, bool(SEC_SYNTH_H),
                                        CHECKOUT_BLOCKED, _b_row_status)
PREVIEW = not FULL

print("=" * 88)
print("SECTION 8 — HEADLINE RESULTS")
print("=" * 88)
print(f"  data dir                 : {DATA}")
print(f"  stage 06_security        : {stage_status_text('06_security')}")
print(f"  stage 07_security_models : {stage_status_text('07_security_models')}")
print(f"  MODE={MODE} · thresholds provenance={thr_prov_h} · security provenance={sec_prov_h}")
print("  security source          : " + (rel(SEC_DIR_H) if SEC_DIR_H else "none")
      + ("   [SYNTHETIC FIXTURE]" if SEC_SYNTH_H else ""))
_qc_h = None
if SEC_DIR_H is not None:
    try:
        _qp_h = SEC_DIR_H / "SUMMARY.json"
        if _qp_h.exists():
            _qc_h = json.loads(_qp_h.read_text(encoding="utf-8")).get("covariate_quality")
    except Exception:
        _qc_h = None
if _qc_h is None:
    print("  covariate quality        : n/a (no security SUMMARY.json)")
else:
    print("  covariate quality        : {} player(s) / {} value(s) outside the declared "
          "bounds; {} effort row(s) excluded (covariate_quality.csv)".format(
              int(_qc_h.get("n_players_flagged") or 0), int(_qc_h.get("n_values_flagged") or 0),
              int(_qc_h.get("n_effort_rows_excluded") or 0)))
    if int(_qc_h.get("n_players_flagged") or 0):
        print("  NOTE                     : 04_models/05_validate consume the same covariate table "
              "(height/weight/bmi are controls),")
        print("                             so their checkpoints predate these exclusions — rebuild "
              "them before quoting their numbers:")
        print("                             FORCE_STAGES=['04_models','05_validate'] (or FORCE=True).")
print("=" * 88)

if not FULL:
    if CHECKOUT_BLOCKED:
        _why = [f"checkout guard blocked the run: {CHECKOUT['stop_message']}"]
        _todo = [CHECKOUT["stop_message"]]
    else:
        _why, _todo = security_chain_issues()
        _why = list(_why)
        if MODE != "full":
            _why.insert(0, f'MODE="{MODE}" → this is not a full-data run')
        if thr_prov_h != "FULL":
            _why.append(f"outputs/thresholds.json provenance={thr_prov_h} → thresholds are not FULL")
        if _b_row_status == "FAILED":
            _why.append("b_primary.csv's first row is block_status=FAILED — no primary result "
                        "(the stage crashed; supplying data will not help)")
        if SEC_SYNTH_H:
            _why.append(f"no real security estimates were found, so §8 would read {rel(SEC_DIR_H)} — "
                        "a committed fixture tuned to produce a plausible effect")
        if not _why:
            _why = ["security provenance is not FULL"]
    _action = ("; then ".join(_todo) if _todo else
               "produce FULL-provenance 06_security + 07_security_models outputs, then re-run this section")
    print()
    _crash = any(str(_r).lower().startswith(("failed", "crashed")) for _r in _why)
    print("!" * 88)
    if CHECKOUT_BLOCKED:
        print("NO RESULT — the checkout guard blocked the run (stale or incomplete checkout)")
    elif _crash:
        print("NO RESULT — the security-model stage FAILED (a code failure, not missing data)")
    else:
        print("NO RESULT — synthetic/sample data in use")
    print("!" * 88)
    if CHECKOUT_BLOCKED:
        print("  No numeric claim is printed: the pipeline did not run against this checkout.")
    elif _crash:
        print("  This is a crash: the stage raised an exception (traceback tail above). Re-supplying")
        print("  data cannot fix it — the data were read; the code that consumes them failed.")
        print("  No numeric claim is printed: block B did not produce a result on this run.")
    else:
        print("  The headline block is WITHHELD ON PURPOSE. Nothing in this section may be quoted as a")
        print("  finding: the fixture that would fill it is tuned to produce a plausible effect.")
    print()
    print("  WHY (specific):")
    for _r in _why:
        print(f"    • {_r}")
    print()
    print(f"  WHAT TO DO: {_action}.")
    print("!" * 88)

print("\nHYPOTHESIS (pre-declared)")
print("  H1: pre-Combine draft stock slipping (moved_down) => a SMALLER Combine-to-game gap than")
print("      high/secure players, i.e. moved_down realises more of its Combine ceiling in games.")
print("  Sign convention: group_coef is moved_down minus high_secure on log game speed.")
print("  POSITIVE group_coef = moved_down closer to its ceiling = SUPPORTS H1; negative = AGAINST.")

def _num(x):
    try:
        v = float(x)
        return v if np.isfinite(v) else np.nan
    except (TypeError, ValueError):
        return np.nan

def _f(x, nd=4):
    v = _num(x)
    return f"{v:.{nd}f}" if np.isfinite(v) else "n/a"

def _i(x):
    v = _num(x)
    return str(int(round(v))) if np.isfinite(v) else "n/a"

def _p(x):
    v = _num(x)
    if not np.isfinite(v):
        return "n/a"
    return "<0.0001" if v < 0.0001 else f"{v:.4f}"

def _truthy(x):
    return str(x).strip().lower() in ("true", "1", "yes")

def _col(r, name):
    return _num(r[name]) if (r is not None and name in r.index) else np.nan



def _headline_block():
    print("\nHEADLINE — WR primary contrast (spec = primary_env_vmax)")
    row = None
    if b_df is None:
        print(f"  not available: {b_err}")
    elif not {"scope", "spec"}.issubset(b_df.columns):
        print("  not available: b_primary.csv lacks the scope/spec columns")
    else:
        _sel = b_df[(b_df["scope"].astype(str) == "WR")
                    & (b_df["spec"].astype(str) == "primary_env_vmax")]
        if not len(_sel):
            print("  not available: no WR row with spec=primary_env_vmax in b_primary.csv")
        else:
            row = _sel.iloc[0]
            print("  n_obs={}  n_players={}  (moved_down={}, high_secure={})  thin={}".format(
                _i(_col(row, "n_obs")), _i(_col(row, "n_players")),
                _i(_col(row, "n_players_moved_down")), _i(_col(row, "n_players_high_secure")),
                _truthy(row["thin"]) if "thin" in row.index else "n/a"))
            print("  group_coef={}  se={}  p={}  95% CI [{}, {}]".format(
                _f(_col(row, "group_coef")), _f(_col(row, "group_se")), _p(_col(row, "group_p")),
                _f(_col(row, "group_ci_lo")), _f(_col(row, "group_ci_hi"))))
            print("  group_coef_corrected={}  se={}  p={}   (ICC-attenuation corrected)".format(
                _f(_col(row, "group_coef_corrected")), _f(_col(row, "group_se_corrected")),
                _p(_col(row, "group_p_corrected"))))
            print("  speed_coef={}  (Combine ceiling to game speed; not a hypothesis test)".format(
                _f(_col(row, "speed_coef"))))

    g, g_p = _col(row, "group_coef"), _col(row, "group_p")
    g_lo, g_hi = _col(row, "group_ci_lo"), _col(row, "group_ci_hi")
    g_c = _col(row, "group_coef_corrected")
    n_md, n_hs = _col(row, "n_players_moved_down"), _col(row, "n_players_high_secure")
    thin = _truthy(row["thin"]) if (row is not None and "thin" in row.index) else True

    verdict, verdict_why = None, ""
    if row is not None:
        if thin or not np.isfinite(n_md) or not np.isfinite(n_hs) or n_md < 15 or n_hs < 15:
            verdict = "NO TEST (thin group)"
            verdict_why = f"thin={thin}; moved_down={_i(n_md)}, high_secure={_i(n_hs)} (min 15 each)"
        elif not (np.isfinite(g) and np.isfinite(g_p)):
            verdict = "not available"
            verdict_why = "group_coef or group_p is blank in b_primary.csv"
        elif g > 0 and g_p < 0.05 and g_lo > 0 and np.isfinite(g_c) and g_c > 0:
            verdict = "SUPPORTED"
            verdict_why = "group_coef>0, p<0.05, CI lower>0, and corrected coefficient >0"
        elif g < 0 and g_p < 0.05 and g_hi < 0:
            verdict = "AGAINST"
            verdict_why = "group_coef<0, p<0.05, CI upper<0"
        else:
            verdict = "INCONCLUSIVE (CI includes zero)"
            verdict_why = "the pre-declared SUPPORTED / AGAINST conditions are not all met"
        if PREVIEW:
            print("\nVERDICT: suppressed — NO RESULT (synthetic/sample data in use).")
        elif verdict == "not available":
            print(f"\nVERDICT: not available — {verdict_why}")
        else:
            print(f"\nVERDICT: {verdict}")
            print(f"  rule: {verdict_why}")

    print("\nEFFECT SIZE")
    if row is None or not np.isfinite(g):
        print("  not available: group_coef is blank/missing")
    else:
        _pct = (np.exp(g) - 1.0) * 100.0
        _dir = "higher" if _pct > 0 else ("lower" if _pct < 0 else "equal")
        print(f"  exp(group_coef) = {np.exp(g):.4f}: at a given Combine ceiling (same controls),")
        print(f"  moved_down in-game top-k speed is {_pct:+.1f}% {_dir} than high_secure.")

    if m_df is None:
        print(f"  group means not available: {m_err}")
    elif not {"scope", "group", "mean_gap", "mean_utilization"}.issubset(m_df.columns):
        print("  group means not available: b_group_means.csv lacks scope/group/mean_gap/mean_utilization")
    else:
        _mm = m_df[(m_df["scope"].astype(str) == "WR")
                   & m_df["group"].astype(str).isin(["moved_down", "high_secure"])]
        _gm = {str(r["group"]): r for _, r in _mm.iterrows()}
        if set(_gm) != {"moved_down", "high_secure"}:
            print("  group means not available: WR moved_down/high_secure rows missing")
        else:
            _gap_md, _gap_hs = _num(_gm["moved_down"]["mean_gap"]), _num(_gm["high_secure"]["mean_gap"])
            _u_md, _u_hs = _num(_gm["moved_down"]["mean_utilization"]), _num(_gm["high_secure"]["mean_utilization"])
            print(f"  WR raw means — moved_down: mean gap {_f(_gap_md)}, mean utilisation {_f(_u_md)};")
            print(f"                  high_secure: mean gap {_f(_gap_hs)}, mean utilisation {_f(_u_hs)}.")
            if all(np.isfinite([_gap_md, _gap_hs, _u_md, _u_hs])):
                print(f"  Interpretation: moved_down sits {abs(_gap_md):.3f} log-units below its ceiling "
                      f"vs {abs(_gap_hs):.3f} for high_secure, and uses {_u_md:.1%} vs {_u_hs:.1%} of it.")

    print("\nFOUR CHECKS" + ("  (preview data — layout check only)" if PREVIEW else ""))

    def _line(label, status, reason):
        print(f"  [{status:<4}] {label}: {reason}")

    # 1. group size
    if row is None:
        _line("1. Group size", "N-A", "no WR primary row to test")
    else:
        _ok1 = (not thin) and np.isfinite(n_md) and np.isfinite(n_hs) and n_md >= 15 and n_hs >= 15
        _line("1. Group size", "PASS" if _ok1 else "FLAG",
              f"moved_down={_i(n_md)}, high_secure={_i(n_hs)} (min 15 each); thin={thin}")

    # 2. sign stability across the WR B specs
    _WANTED = ["primary_env_vmax", "robust_peak_speed", "robust_speed_at_20yd",
               "panel_trend_only", "panel_level_fallback", "topk_mean_player"]
    if b_df is None or not np.isfinite(g):
        _line("2. Sign stability", "N-A", "no primary coefficient / b_primary.csv missing")
    elif not {"scope", "spec", "group_coef"}.issubset(b_df.columns):
        _line("2. Sign stability", "N-A", "b_primary.csv lacks scope/spec/group_coef")
    else:
        _wr = b_df[b_df["scope"].astype(str) == "WR"].drop_duplicates("spec")
        _found = _wr["spec"].astype(str).tolist()
        _missing = [s for s in _WANTED if s not in _found]
        _sub = _wr[_wr["spec"].astype(str).isin(_WANTED)]
        _cs = pd.to_numeric(_sub["group_coef"], errors="coerce").dropna()
        _agree = int((np.sign(_cs) == np.sign(g)).sum())
        _dirword = "positive" if g > 0 else "negative"
        if _missing:
            _line("2. Sign stability", "FLAG",
                  f"missing WR spec(s): {', '.join(_missing)}; {_agree}/{len(_WANTED)} agree on {_dirword}")
        elif len(_cs) == len(_WANTED) and _agree == len(_WANTED):
            _line("2. Sign stability", "PASS",
                  f"{_agree}/{len(_WANTED)} WR specs share the {_dirword} sign "
                  "(3 speed features + trend_only + level_fallback + topk_mean)")
        else:
            _line("2. Sign stability", "FLAG",
                  f"{_agree}/{len(_WANTED)} WR specs share the {_dirword} sign ({len(_cs)} estimated)")

    # 3. dose-response in c_end.csv
    if c_df is None:
        _line("3. Dose-response", "N-A", c_err)
    elif not {"scope", "basis", "percentile", "fitted_gap"}.issubset(c_df.columns):
        _line("3. Dose-response", "N-A", "c_end.csv lacks scope/basis/percentile/fitted_gap")
    elif row is None or not np.isfinite(g):
        _line("3. Dose-response", "N-A", "no primary group_coef to set the expected direction")
    else:
        _cc = c_df[(c_df["scope"].astype(str) == "WR")
                   & (c_df["basis"].astype(str) == "mean_ceiling")].copy()
        _pc = pd.to_numeric(_cc["percentile"], errors="coerce")
        if _pc.notna().any() and float(_pc.max()) > 1.0:
            _pc = _pc / 100.0
        _cc["_pc"] = _pc
        _cc = _cc.dropna(subset=["_pc"])
        def _gap_at(v):
            _r = _cc[(_cc["_pc"] - v).abs() < 1e-6]
            return _num(_r["fitted_gap"].iloc[0]) if len(_r) else np.nan
        _g10, _g50, _g90 = _gap_at(0.10), _gap_at(0.50), _gap_at(0.90)
        if not all(np.isfinite([_g10, _g50, _g90])):
            _line("3. Dose-response", "N-A", "percentiles 10/50/90 not all present for WR mean_ceiling")
        else:
            if g > 0:
                _dir_ok, _mono = _g90 > _g10, _g10 < _g50 < _g90
            else:
                _dir_ok, _mono = _g90 < _g10, _g10 > _g50 > _g90
            _line("3. Dose-response", "PASS" if (_dir_ok and _mono) else "FLAG",
                  f"fitted gap p10={_g10:+.4f}, p50={_g50:+.4f}, p90={_g90:+.4f}; "
                  f"p90-p10={_g90 - _g10:+.4f}; expected {'increasing' if g > 0 else 'decreasing'}; "
                  f"monotone={'yes' if _mono else 'no'}")

    # 4. falsification placebos + effort contrast
    if f_df is None:
        _line("4. Falsification", "N-A", f_err)
    elif not {"test", "term", "coef", "p"}.issubset(f_df.columns):
        _line("4. Falsification", "N-A", "falsification.csv lacks test/term/coef/p")
    else:
        _pl = f_df[(f_df["test"].astype(str) == "placebo")
                   & f_df["term"].astype(str).isin(["height", "weight"])]
        if not len(_pl):
            _line("4. Falsification", "N-A", "no height/weight placebo rows")
        else:
            _bad, _bits = [], []
            for _, _r in _pl.iterrows():
                _c, _pv = _num(_r["coef"]), _num(_r["p"])
                _bits.append(f"{_r['term']} coef={_f(_c)} p={_p(_pv)}")
                if (np.isfinite(_c) and np.isfinite(_pv) and _pv < 0.05
                        and np.isfinite(g) and np.sign(_c) == np.sign(g)):
                    _bad.append(str(_r["term"]))
            if _bad:
                _line("4. Falsification", "FLAG",
                      "; ".join(_bits) + f" — {', '.join(_bad)} reproduce(s) the group sign at p<0.05")
            else:
                _line("4. Falsification", "PASS",
                      "; ".join(_bits) + " — neither placebo reproduces the group pattern")
        _eff = f_df[f_df["test"].astype(str) == "effort_split"]
        if len(_eff):
            _parts = [f"{_r['stratum']}: coef={_f(_num(_r['coef']))}, p={_p(_num(_r['p']))}"
                      for _, _r in _eff.iterrows()]
            print("       effort contrast — " + "; ".join(_parts))

    print("\nCAVEATS")
    _icc = _col(row, "reliability_icc")
    _icc_txt = _f(_icc, 2) if np.isfinite(_icc) else "not available"
    print(f"  • Attenuation: env_vmax reliability ICC is {_icc_txt}. The naive group_coef is")
    print("    attenuated toward zero by measurement error; the corrected coefficient is shown beside it.")
    print("    Thin strata (check 1) also widen every interval.")
    print("  • Combine peaking: the F markers capture peaking. A NEGATIVE sign has two readings:")
    print("    (i) weaker incentive to invest in the speed mechanism, or (ii) secure players peaking")
    print("    harder at the Combine, which inflates their ceiling and makes their game gap look larger.")
    print("    The design cannot separate those two on its own.")
    if i_df is None:
        print(f"  • Selection: not available ({i_err}).")
    elif not {"weighting", "group_coef"}.issubset(i_df.columns):
        print("  • Selection: not available (e_ipw.csv lacks weighting/group_coef).")
    else:
        _note = (i_df["note"].astype(str) if "note" in i_df.columns
                 else pd.Series([""] * len(i_df), index=i_df.index))
        _un = i_df[i_df["weighting"].astype(str) == "unweighted"]
        _ipw = i_df[(i_df["weighting"].astype(str) == "ipw")
                    & _note.str.contains("primary", case=False)]
        if not len(_ipw):
            _ipw = i_df[i_df["weighting"].astype(str) == "ipw"]
        if len(_un) and len(_ipw):
            _a, _bb = _num(_un["group_coef"].iloc[0]), _num(_ipw["group_coef"].iloc[0])
            _rel = ((_bb - _a) / abs(_a) * 100.0) if (np.isfinite(_a) and _a != 0) else np.nan
            if not np.isfinite(_a) or not np.isfinite(_bb):
                _move = "could not be compared (blank coefficient)"
            elif abs(_a) < 0.02:
                _move = "does not move the estimate materially (baseline near zero)"
            elif np.isfinite(_rel) and abs(_rel) >= 5.0:
                _move = "moves the estimate"
            else:
                _move = "does not move the estimate materially"
            print(f"  • Selection: unweighted {_f(_a)} -> IPW {_f(_bb)} ({_rel:+.1f}%); IPW {_move}.")
            print("    Only drafted players have game data, so selection on being drafted remains a threat.")
        else:
            print("  • Selection: not available (no unweighted/IPW rows).")

    print("\nWHAT THIS IS NOT")
    print("  No causal claim and no claim about intent or effort — incentive exposure only.")


if FULL:
    _headline_block()
elif SHOW_SYNTHETIC_PREVIEW and not CHECKOUT_BLOCKED:
    print()
    print("-" * 88)
    print("SYNTHETIC LAYOUT PREVIEW — NOT DATA")
    print("  Rendered only because SHOW_SYNTHETIC_PREVIEW=True. Every number below comes from the")
    print("  committed synthetic fixture, which is tuned to look plausible. It is NOT a finding.")
    print("-" * 88)
    _headline_block()
else:
    print()
    print("  Layout preview suppressed — set SHOW_SYNTHETIC_PREVIEW=True (Configuration) to render it.")


## 9. Route robustness — can route mix explain the group contrast?

**Robustness — not a pre-declared test.** This section sits *outside* §8 on purpose: it reads
`outputs/security_models/route_robustness/` (written by `src/08_route_robustness.py`) and re-estimates
block B's exact specification on route strata, then resamples players. It cannot change
`b_primary.csv`, the verdict, or any pre-declared number. If the directory is absent it says so
instead of substituting anything.

What it checks: `moved_down` receivers may simply run a different route menu (fewer deep routes,
more decoy duty), which would lower a top-k *peak-speed* outcome mechanically. The block prints
(A) the balance table, (B) the within-stratum coefficients, and (C) leave-one-player-out and
player-level permutation p-values.

In [ ]:
# ===================== 9. Robustness — route mix (not a pre-declared test) =====================
# Reads outputs/security_models/route_robustness/ (src/08_route_robustness.py).
# NOT part of §8: this block cannot change the verdict, and it does not re-open it.
print("robustness — not a pre-declared test")
print("=" * 88)

RR_DIR = out_path("security_models", "route_robustness")


def _rr(name):
    p = RR_DIR / name
    if not p.exists():
        return None
    try:
        return pd.read_csv(p)
    except Exception as exc:
        print(f"  {name}: unreadable ({type(exc).__name__}: {exc})")
        return None


if not RR_DIR.exists():
    print("  no route_robustness/ output — nothing is estimated here.")
    print("  run: python src/08_route_robustness.py --mode full")
else:
    print(f"  source: {RR_DIR.relative_to(REPO)}")
    bal = _rr("route_balance.csv")
    if bal is not None and len(bal):
        cols = [c for c in ("scope", "group", "n_players", "n_efforts", "share_VERTICAL",
                            "share_BREAK_IN", "share_BREAK_OUT", "share_SHORT",
                            "pass_length_mean", "pass_length_median", "snap_share_mean",
                            "target_rate") if c in bal.columns]
        print("\nA. Route balance (T3 primary panel)")
        print(bal[bal.panel == "primary_env_vmax"][cols].to_string(
            index=False, float_format=lambda v: f"{v:.3f}"))
        t = _rr("route_balance_tests.csv")
        if t is not None and len(t):
            print("\n   tests:")
            for _, r in t.iterrows():
                print(f"   {r['scope']}: chi2 p={r['chi2_p']:.3f} (V={r['cramers_v']:.2f}); "
                      f"depth Welch p={r['pass_length_welch_p']:.3f}; "
                      f"target-rate diff={r['target_rate_diff']:+.1%} (p={r['target_diff_p']:.3f})")

    comp = _rr("composition.csv")
    if comp is not None and len(comp):
        cols = [c for c in ("panel", "scope", "n_obs", "n_players_moved_down",
                            "n_players_high_secure", "thin", "group_coef", "group_p",
                            "note") if c in comp.columns]
        print("\nA2. Composition (is the pooled number a WR number?)")
        print(comp[cols].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    strata = _rr("route_strata.csv")
    if strata is not None and len(strata):
        cols = [c for c in ("panel", "scope", "stratum", "n_obs", "n_players_moved_down",
                            "n_players_high_secure", "thin", "group_coef", "group_se",
                            "group_p", "note") if c in strata.columns]
        print("\nB. Within-stratum re-estimation (same spec as block B)")
        print(strata[cols].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    loo = _rr("loo_summary.csv")
    if loo is not None and len(loo):
        print("\nC. Leave-one-player-out")
        for _, r in loo.iterrows():
            print(f"   {r['panel']}/{r['scope']}: coef {r['baseline_coef']:+.4f} "
                  f"(p={r['baseline_p']:.4f}); LOO coef {r['coef_min']:+.4f}…{r['coef_max']:+.4f}; "
                  f"p {r['p_min']:.4f}…{r['p_max']:.4f}; "
                  f"share p<0.05 = {r['share_p_lt_05']:.0%}")
    perm = _rr("permutation.csv")
    if perm is not None and len(perm):
        print("\nD. Player-level permutation")
        for _, r in perm.iterrows():
            print(f"   {r['panel']}/{r['scope']}: observed {r['observed_coef']:+.4f}; "
                  f"permutation p={r['p_perm_two_sided']:.4f} "
                  f"({int(r['n_reps'])} reps; directional p={r['p_perm_directional']:.4f})")
    print("\n   This block is descriptive robustness only — §8 remains the verdict.")

## 10. Draft outcomes and targeted routes — two exploratory probes

**Exploratory — never merged into §8.** This section runs `src/09_draft_and_target.py` **in the
notebook kernel** (via `stage_runner`, so it always executes the code on disk) and prints its
tables. It asks two questions that the pre-registered verdict does not answer:

* **Part A — realised draft capital.** Do the `moved_down` and `high_secure` sort populations end
  up with different actual draft slots? Does the group coefficient survive adding
  `log(draft_overall_pick)` as a covariate, or within declared draft bands? Draft slot is
  **post-Combine**, so this is a mechanism/diagnostic probe, **not** a clean adjustment:
  conditioning on it can open a collider. A group difference is both evidence the pre-Combine sort
  key is real and a live confound (worse draft capital → worse QB/team/opportunity).
* **Part B — targeted routes.** The target flag is derived by one key join from `efforts.csv` on
  (`nfl_id`, `game_id`, `play_id`) and the contrast is refitted on `target == True`,
  `target == False` and all rows. Being targeted is a post-treatment, role-dependent variable, so
  target-only is **not** a clean estimate. The split is a decomposition: the non-target (decoy)
  subset is where the mechanical "peak speed is an opportunity-limited maximum" suppression should
  live.

Outputs go to `outputs/security_models/draft_and_target/`. If the full-data inputs are absent, the
cell prints **exactly which files are missing** and estimates nothing — it never substitutes sample
or synthetic numbers. No command line is required.


In [ ]:
# ===================== 10. Exploratory — draft outcomes & targeted routes =====================
# Runs src/09_draft_and_target.py IN THIS KERNEL. NOT part of §8: it cannot change the verdict.
# If the full-data inputs are absent it prints the missing files and estimates nothing.
print("exploratory — not a pre-declared test; never merged into §8")
print("=" * 88)

DT_DIR = out_path("security_models", "draft_and_target")
_dt_stage = stage_runner.load_stage_module("09_draft_and_target")[0]
_missing_dt = _dt_stage.required_inputs(RUN_CFG, False, "treated")

if _missing_dt:
    print("  §10 NOT RUN — missing full-data input(s):")
    for _label, _p in _missing_dt:
        print(f"    - {_label}: {_p}")
    print("  Nothing is estimated from substitutes; run the full pipeline above first.")
elif MODE != "full":
    print(f"  inputs are present, but MODE={MODE!r}. §10 is a full-data section and is not run "
          "on a sample.")
else:
    _dt_res = stage_runner.run_stage("09_draft_and_target",
                                     ["--mode", MODE, "--config", str(RUN_CFG_PATH)],
                                     outputs_root=OUTPUTS)
    print(f"  stage ok={_dt_res['ok']} ({_dt_res['seconds']}s)  code {_dt_res.get('action', '')}")
    if not _dt_res["ok"]:
        print(f"  error: {_dt_res['error']}")
        for _ln in _dt_res["tail"]:
            print(f"  | {_ln}")

    def _dt_read(name):
        _p = DT_DIR / name
        if not _p.exists():
            print(f"  {name}: MISSING")
            return None
        try:
            return pd.read_csv(_p)
        except Exception as _exc:
            print(f"  {name}: unreadable ({type(_exc).__name__}: {_exc})")
            return None

    if not DT_DIR.exists():
        print("  no draft_and_target/ output — nothing is reported here.")
    else:
        print(f"  source: {DT_DIR.relative_to(REPO)}")
        _tables = [
            ("A1. Draft outcomes by group", "draft_by_group.csv",
             ["panel", "scope", "group", "n_players_assigned", "n_matched", "join_rate",
              "median_overall_pick", "pick_q1", "pick_q3", "share_R1", "share_R2",
              "share_R3-4", "share_R5-7", "share_R1_2", "thin"]),
            ("A2. Draft-slot tests", "draft_tests.csv",
             ["panel", "scope", "analysis", "n_moved_down", "n_high_secure", "u", "chi2",
              "odds_ratio", "rank_biserial", "p", "note"]),
            ("A3. Sort-key validity (Spearman vs actual pick)", "draft_spearman.csv",
             ["scope", "exposure", "n_players", "rho", "p"]),
            ("A4. Mechanism probe (covariate / draft-band fits)", "draft_conditional_fits.csv",
             ["panel", "spec", "scope", "model", "n_obs", "n_players_moved_down",
              "n_players_high_secure", "group_coef", "group_se", "group_p", "thin", "note"]),
            ("B1. Target-subset fits", "target_subset_fits.csv",
             ["panel", "spec", "scope", "subset", "target_rate", "n_obs",
              "n_players_moved_down", "n_players_high_secure", "group_coef", "group_se",
              "group_p", "thin", "note"]),
            ("B2. Target balance", "target_balance.csv",
             ["panel", "spec", "scope", "subset", "group", "n_obs", "n_players", "target_rate",
              "n_target_missing"]),
        ]
        for _title, _name, _cols in _tables:
            _df = _dt_read(_name)
            if _df is None or not len(_df):
                continue
            _use = [c for c in _cols if c in _df.columns]
            print(f"\n{_title}")
            print(_df[_use].to_string(index=False, float_format=lambda v: f"{v:.4f}"))
        print("\n  This block is exploratory only — §8 remains the verdict.")


## 11. Season & panel scope — the 2×2 grid

**D49.** The primary specification is **REG+POST × all_seasons**. This section shows the other
three cells as sensitivities:

| | all_seasons | rookie_only |
|---|---|---|
| **REG+POST** | **PRIMARY** | sensitivity |
| **REG+POST+PRE** | sensitivity | sensitivity |

**Why a panel axis.** `rookie_only` uses only the season equal to the player's draft year, so the
Combine ceiling is contemporaneous with the game data and every draft class is at the same career
stage. It removes survivorship conditioning on later seasons. Its cost is fewer clusters (tenure
is unavailable there **by construction** — `years_since_combine` is identically 0 and is dropped
as a constant control, reported in `dropped_controls` and below). The all-seasons cell is the
primary; the rookie cell is a declared sensitivity, never a replacement.

**One build, four cells.** Stage 03 carried `season`/`season_type` on every effort and top-k per
season scope. The panel filter is model-stage only: it filters rows **and** recomputes each
player's snap count inside the panel, so `min_snaps` and `snap_share` never use career totals in
the rookie cell. Nothing is rebuilt from the game-tracking files. Every output row carries
`season_scope` and `panel`, and `scope_grid.csv` labels all four cells distinctly.

> Sensitivity runs write under `security_models/<cell>/` and never overwrite the primary
> `b_primary.csv`.


In [ ]:
# ===================== 11. Season & panel scope — the 2x2 grid (D49) =====================
# Runs the three non-primary cells of 07 in-notebook, then prints the 2x2 table with cluster
# counts. No command line: everything goes through stage_runner, so the code on disk is what runs.
print("scope grid — primary is REG+POST x all_seasons")
print("=" * 88)

SEC_BASE = (out_path("security_models") if MODE == "full"
            else out_path("security_models", "synthetic", "planted"))
PRIMARY_DIR = SEC_BASE  # config defaults: active scope=primary, active panel=all_seasons
CELL_DIRS = {
    ("all_seasons", "primary"): SEC_BASE,
    ("all_seasons", "sensitivity"): SEC_BASE / "sensitivity",
    ("rookie_only", "primary"): SEC_BASE / "rookie_only",
    ("rookie_only", "sensitivity"): SEC_BASE / "rookie_only_sensitivity",
}

if CHECKOUT_BLOCKED:
    print("  §11 skipped — checkout guard blocked")
elif not (PRIMARY_DIR / "b_primary.csv").exists():
    print("  §11 NOT RUN — primary outputs absent")
    print(f"    - {PRIMARY_DIR / 'b_primary.csv'}")
    print("  Run §3 first; nothing is estimated from substitutes.")
else:
    for (panel, scope), target in CELL_DIRS.items():
        if (panel, scope) == ("all_seasons", "primary"):
            continue  # already produced by §3
        args = (["--mode", MODE, "--panel", panel, "--scope", scope,
                 "--config", str(RUN_CFG_PATH)]
                if MODE == "full" else
                ["--mode", MODE, "--synthetic", "--synthetic-regime", "planted",
                 "--panel", panel, "--scope", scope, "--config", str(RUN_CFG_PATH)])
        res = stage_runner.run_stage("07_security_models", args, outputs_root=OUTPUTS)
        print(f"  {panel} x {scope}: ok={res['ok']} ({res['seconds']:.2f}s)"
              + (f"  error: {res['error']}" if not res["ok"] else ""))
        for line in (res["tail"] if not res["ok"] else []):
            print(f"    {line}")

    print()
    print("  scope_grid.csv — each cell's snapshot/eligibility denominator:")
    grid = read_csv(PRIMARY_DIR / "scope_grid.csv")
    if len(grid):
        cols = [c for c in ("scope_cell", "panel", "season_types", "is_primary",
                            "n_route_snaps", "n_players", "n_eligible", "n_model_rows",
                            "n_players_moved_down", "n_players_high_secure",
                            "median_n_route_snaps", "mean_snap_share", "thin") if c in grid.columns]
        print(grid[cols].to_string(index=False))
    else:
        print("    scope_grid.csv missing")

    print()
    print("  2x2 B contrast (spec=primary_env_vmax) — clusters are distinct players:")
    rows = []
    for (panel, scope), d in CELL_DIRS.items():
        b = read_csv(d / "b_primary.csv")
        if not len(b):
            continue
        b = b[(b["spec"] == "primary_env_vmax") & (b["scope"] == "pooled")]
        if not len(b):
            continue
        r = b.iloc[0]
        rows.append(dict(
            cell=f"{panel}|{scope}" + ("  <-- PRIMARY" if (panel, scope) == ("all_seasons", "primary") else ""),
            season_scope=r.get("season_scope", ""), panel=r.get("panel", panel),
            n_obs=r.get("n_obs"), n_players=r.get("n_players"),
            n_md=r.get("n_players_moved_down"), n_hs=r.get("n_players_high_secure"),
            group_coef=r.get("group_coef"), group_se=r.get("group_se"), group_p=r.get("group_p"),
            thin=r.get("thin"), tenure=("years_since_combine dropped (constant)"
                                        if "years_since_combine:constant"
                                        in str(r.get("dropped_controls", "")) else ""),
        ))
    if rows:
        print(pd.DataFrame(rows).to_string(index=False))
        print()
        print("  read: positive group_coef = moved_down has a SMALLER Combine→game gap.")
        print("  Quote the PRIMARY cell. The other three are declared sensitivities; a thin")
        print("  cell is reported as thin and is not a result.")
    else:
        print("    no fitted b_primary.csv found in any cell")

print()
print("  NOTE: the sample/fixture numbers only prove the code path. The real four-cell grid")
print("  needs one full rebuild; the fixture's roster is not the NFL population.")


## 12. Dose-response — continuous exposure (exploratory, post hoc)

**Exploratory — post hoc.** The pre-registered exposure is binary (`moved_down` iff `edp_trend >= +3.0`),
and moving that cut to +5 relabels only a few players while moving *p* a lot — the threshold is not doing
real work. `src/10_dose_response.py` (D50) therefore models the exposure **continuously**: a linear slope,
a 3-df restricted cubic spline, an explicit knot at 5, fixed bins (`<=0 / 1-2 / 3-4 / 5-9 / 10+`, plus a
quantile variant when a bin is thin) and a segmented (kink) model — under seven **capital** treatments.

This section **cannot** change §8, `b_primary.csv` or any frozen value. It is read-only display of
`outputs/security_models/dose/`.

**How to read the sign.** `coef` = the change in log(game peak speed) per **+1 EDP spot fallen**, holding
the Combine ceiling fixed. **NEGATIVE = a slipping player realises LESS of his ceiling** (the PI's stated
expectation); **POSITIVE** is the direction the pre-registered `TASK.md` H1 *wording* predicts. The
discrepancy is documented in D50 and stamped on every row (`sign_convention`). `edp_trend` positive =
the EDP number rose = the stock fell.

**Capital is mandatory.** If the slope dies once `overall_pick` / the round band is controlled, the finding
is that *the dose slope is draft position in disguise* — the panel says so in one sentence.

**The figure is supplementary.** It is **not** one of the 5 writeup figures (`FIGURES` is untouched, the
`deliverables.figures_max: 5` cap still holds). With no FULL output this cell prints a `NO RESULT` block and
invents nothing (D41); set `SHOW_SYNTHETIC_PREVIEW = True` for a labelled **SYNTHETIC LAYOUT PREVIEW — NOT
DATA**. A FULL run saves the figure to `outputs/figures/fig_dose_response.png`; otherwise it goes to
`outputs/figures/synthetic/` with `_SYNTHETIC_PREVIEW` in the filename.

In [ ]:
# ===================== 12. Dose-response — continuous exposure (EXPLORATORY) =====================
# Reads outputs/security_models/dose/ (src/10_dose_response.py, D50).
# NOT part of §8: this block cannot change the verdict, and the figure is NOT one of the 5 writeups.
print("dose-response — exploratory / post hoc (D50)")
print("=" * 88)

DOSE_DIR = out_path("security_models", str(CFG_REPO["dose"]["out_subdir"]))
DOSE_SYNTH = DOSE_DIR / "synthetic" / "dose"


def _dose_read(name, d=None):
    p = Path(d or DOSE_DIR) / name
    if not p.exists():
        return None
    try:
        return pd.read_csv(p)
    except Exception as exc:
        print(f"  {name}: unreadable ({type(exc).__name__}: {exc})")
        return None


def _dose_source():
    """(dir, is_synthetic, why): the FULL dose panel wins, else the committed fixture."""
    fits = _dose_read("dose_fits.csv")
    if fits is not None and len(fits) and "provenance" in fits.columns:
        prov = str(fits["provenance"].iloc[0])
        if prov == "FULL":
            return DOSE_DIR, False, ""
        return DOSE_SYNTH, True, f"dose panel provenance is '{prov}'"
    return DOSE_SYNTH, True, f"no dose panel at {rel(DOSE_DIR)}"


DOSE_SRC, DOSE_IS_SYNTH, DOSE_WHY = _dose_source()

if DOSE_IS_SYNTH and not SHOW_SYNTHETIC_PREVIEW:
    print("NO RESULT — no FULL dose-response panel is available.")
    print(f"  reason: {DOSE_WHY}.")
    print("  Nothing is rendered here: synthetic/sample numbers are withheld (D41), because a")
    print("  dose curve read out of a fixture would look like a finding.")
    print("  run: python src/10_dose_response.py --mode full        (needs the full game files)")
    print("  or set SHOW_SYNTHETIC_PREVIEW = True for a labelled LAYOUT PREVIEW (never a finding).")
else:
    if DOSE_IS_SYNTH:
        print("SYNTHETIC LAYOUT PREVIEW — NOT DATA")
        print(f"  source: {rel(DOSE_SRC)}  ({DOSE_WHY})")
        print("  every number below is generated by src/synthetic_security_game_fixture.py;")
        print("  it proves the layout and the plumbing, nothing about the NFL.")
        if not (DOSE_SRC / "dose_fits.csv").exists():
            _r = run_stage("10_dose_response",
                           ["--synthetic", "--synthetic-regime", "dose",
                            *force_args("10_dose_response")])
            print(f"  built the fixture panel in-kernel: ok={_r['ok']} ({_r['seconds']}s)")
    else:
        print(f"  source: {rel(DOSE_SRC)}  (FULL)")
    print(f"  sign convention: {CFG_REPO['dose']['sign_convention']}")

    fits = _dose_read("dose_fits.csv", DOSE_SRC)
    if fits is None or not len(fits):
        print("  dose_fits.csv is absent — nothing to print.")
    else:
        fits["coef"] = to_num(fits["coef"])
        fits["p"] = to_num(fits["p"])
        keep = [c for c in ("season_scope", "scope", "feature", "capital_treatment", "term",
                            "coef", "se", "p", "ci_lo", "ci_hi", "n_obs", "n_players",
                            "coef_corrected", "thin") if c in fits.columns]
        print("\nA. Linear slope (env_vmax, all_seasons; every capital treatment)")
        a = fits[(fits.spec == "dose_linear") & (fits.season_scope == "all_seasons")
                 & (fits.feature == "env_vmax") & (fits.term == "slope")]
        order = ["unadjusted", "pick_linear", "round_band", "pick_plus_level",
                 "within_R1", "within_R2_R3", "within_R4plus"]
        a = a.assign(_o=a.capital_treatment.map({t: i for i, t in enumerate(order)}))
        print(a.sort_values(["scope", "_o"])[keep].to_string(
            index=False, float_format=lambda v: f"{v:.4g}"))
        print("\nB. Spline / explicit-knot overall tests (Wald over the dose basis, clustered)")
        b = fits[fits.spec.isin(["dose_spline", "dose_spline_knot5"]) & (fits.term == "overall")
                 & (fits.season_scope == "all_seasons") & (fits.capital_treatment == "unadjusted")]
        bcols = [c for c in ("scope", "spec", "feature", "test_stat", "df", "p", "n_obs",
                             "n_players") if c in b.columns]
        print(b[bcols].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
        print("  (notes in dose_fits.csv name the knots and any dropped control)")
        print("\nC. Segmented (kink) slopes — slope_above_5 uses the FULL clustered covariance")
        c = fits[(fits.spec == "dose_kink") & (fits.season_scope == "all_seasons")
                 & (fits.capital_treatment == "unadjusted")]
        print(c[[x for x in ("scope", "term", "coef", "se", "p", "ci_lo", "ci_hi") if x in c.columns]]
              .to_string(index=False, float_format=lambda v: f"{v:.4g}"))
        bins = _dose_read("dose_bins.csv", DOSE_SRC)
        if bins is not None and len(bins):
            print("\nD. Bins (WR, all_seasons, unadjusted): counts, means and the contrast vs <=0")
            bb = bins[(bins.scope == "WR") & (bins.season_scope == "all_seasons")
                      & (bins.capital_treatment == "unadjusted")]
            bcols = [x for x in ("binning", "bin", "n_players", "n_obs", "mean_edp_trend",
                                 "mean_gap", "mean_realisation_pct", "ref_contrast", "ref_se",
                                 "thin") if x in bb.columns]
            print(bb[bcols].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
        sj = Path(DOSE_SRC) / "SUMMARY.json"
        if sj.exists():
            _st = json.loads(sj.read_text()).get("statements") or {}
            for key, label in (("sign", "In words — linear"),
                               ("capital", "In words — capital confound"),
                               ("features", "In words — feature robustness"),
                               ("kink", "In words — segmented")):
                if _st.get(key):
                    print(f"\n{label}: {_st[key]}")

    fig_src = Path(DOSE_SRC) / str(CFG_REPO["dose"]["figure"]["file"])
    if not fig_src.exists():
        print("\n  figure missing — nothing is rendered.")
    else:
        fig_dst_dir = FIGDIR if not DOSE_IS_SYNTH else (FIGDIR / "synthetic")
        fig_dst_dir.mkdir(parents=True, exist_ok=True)
        suffix = "" if not DOSE_IS_SYNTH else "_SYNTHETIC_PREVIEW"
        fig_dst = fig_dst_dir / f"fig_dose_response{suffix}.png"
        shutil.copy2(fig_src, fig_dst)
        print(f"\nfigure: {rel(fig_dst)}  ({fig_dst.stat().st_size} bytes, "
              f"{int(CFG_REPO['dose']['figure']['dpi'])} dpi)"
              + ("   — SYNTHETIC PREVIEW, not a finding" if DOSE_IS_SYNTH else ""))
        print("  supplementary exploratory figure — NOT one of the 5 writeup figures")
        try:
            from IPython.display import Image, display
            display(Image(filename=str(fig_dst)))
        except Exception as exc:
            print(f"  (inline display unavailable: {type(exc).__name__}: {exc})")
    print("\n  This block is exploratory/post hoc only — §8 remains the pre-registered verdict.")

## 13. Leverage × ceiling — the current LC chain (D59–D63)

**This is the section that belongs to the current task** (`TASK.md`). Everything below is read from
`outputs/game/lc/route_depth/` — the route-depth stage of the LC chain run in §3. No number here is
typed by hand.

What it prints:

1. **Break coverage** — route snaps analysed, routes with a detected break, breaks with a known
   throw frame, **pre-throw breaks**, and the pre-throw count **per player**.
2. **Coverage by route family** — route snaps, breaks, pre-throw breaks, break rate and median
   break depth per family.
3. **The D63 keep-or-fall-back rule** — whether the model keeps the continuous centred
   `break_depth_yd` control or falls back to D61 option 4 (route family + `break_found` indicator),
   stated as the rule it is, with the value this run computes on each side of it.

**Sample mode never decides anything.** With `MODE="sample"` the section prints the observed counts
stamped `UNVALIDATED SAMPLE OUTPUT`, names the full-data inputs that are missing, and states that the
keep/fall-back decision is **not taken**. `TASK.md` forbids interpreting or tuning on sample numbers
(D57); the decision rule is evaluated on `FULL` provenance only.

**What this section is not.** It reports the *depth control's* coverage, not the estimand. The
effort table, the primary model and the validation stage are §14 and §15 (`04_efforts`,
`05_models`, `06_validation`, D65–D66). No effect size, interval or p-value appears here, by design.


In [ ]:
# ===================== 13. LC chain results — coverage and the D63 rule =====================
# Reads outputs/game/lc/route_depth/ only. Every number is computed from the files on disk.
print("leverage × ceiling — LC chain (D59–D63)")
print("=" * 88)

RD_DIR = out_path("game", "lc", "route_depth")
RD_CSV = RD_DIR / "route_depth.csv"
RD_SUM = RD_DIR / "route_depth_summary.json"
RD_SIG = RD_DIR / "route_depth_sensitivity.csv"

STAMP = "UNVALIDATED SAMPLE OUTPUT"
LABEL = "FULL" if MODE == "full" else STAMP

# ---- the D63 rule: read from config, never typed here ------------------------
rule = (LC.lc(RUN_CFG).get("route_depth") or {}).get("decision_rule") or {}
if not rule:
    print("D63 rule block missing from config (bdb27_lc.route_depth.decision_rule) — "
          "cannot evaluate the keep/fall-back choice; nothing estimated.")
else:
    REQ_N     = int(rule["min_pre_throw_breaks"])
    REQ_FAM   = int(rule["min_families_meeting_bar"])
    REQ_FAM_N = int(rule["min_pre_throw_breaks_per_family"])
    FALLBACK  = str(rule.get("fallback", "D61 option 4"))

    print(f"label  : {LABEL}")
    print(f"source : {rel(RD_DIR)}")
    print()
    print("D63 rule — pre-declared, evaluated on FULL data only:")
    print("  KEEP the continuous centred break_depth_yd iff")
    print(f"    (a) pre-throw breaks (break found, throw frame known, before the throw) >= {REQ_N:,}")
    print("        AND")
    print(f"    (b) at least {REQ_FAM} route families each have >= {REQ_FAM_N} pre-throw breaks")
    print(f"  OTHERWISE fall back to {FALLBACK}: route family + break_found indicator,")
    print("  with depth reported descriptively and never estimated.")

    def _missing_inputs():
        """Exactly which FULL inputs are absent on this machine (never a generic 'no data')."""
        out = list(FULL_TRACKING)
        if not out:
            return ["(none — the full tracking files are present; the stage has not been run yet)"]
        return out

    if not RD_CSV.exists():
        print()
        print("NOT RUN — outputs/game/lc/route_depth/route_depth.csv is absent.")
        print("  missing full-data inputs:")
        for f in _missing_inputs():
            print(f"    - {f}")
        print("  Nothing estimated.")
    else:
        rd = pd.read_csv(RD_CSV, low_memory=False)
        need = {"nfl_id", "game_id", "route_family", "break_found",
                "break_after_throw", "throw_known", "break_depth_yd"}
        absent_cols = sorted(need - set(rd.columns))
        if absent_cols:
            print()
            print(f"CANNOT COMPUTE — route_depth.csv lacks column(s) {absent_cols}. Nothing estimated.")
        else:
            found = is_true(rd["break_found"])
            known = is_true(rd["throw_known"])
            after = is_true(rd["break_after_throw"])
            pre = found & known & ~after              # strict pre-throw definition (D62 §3)
            depth = to_num(rd["break_depth_yd"])

            n_routes = len(rd)
            n_players = int(rd["nfl_id"].nunique())
            n_games = int(rd["game_id"].nunique())
            n_found = int(found.sum())
            n_known = int((found & known).sum())
            n_pre = int(pre.sum())

            # -- 1. coverage ------------------------------------------------------------
            print()
            rec = read_json(RD_SUM) or {}
            prov_rd = str(rec.get("provenance", ""))
            cov = rec.get("coverage") or {}

            print("break coverage (descriptive counts from route_depth.csv)")
            print(f"  route snaps analysed ......... {n_routes:,}  ({n_players} players with tracked"
                  f" snaps, {n_games:,} games)")
            if rec.get("n_route_snaps") is not None:
                key_snaps = int(rec.get("n_route_snaps") or 0)
                key_players = rec.get("n_players")
                key_games = rec.get("n_games")
                no_track = rec.get("n_route_snaps_without_tracking")
                print(f"  route-snap key table ......... {key_snaps:,} snaps, {key_players} players,"
                      f" {key_games} games (before the tracking join:"
                      f" {no_track} snaps had no tracking)")
                print("                                per-player counts below are per player WITH"
                      " tracked snaps, not per route-running player")
            print(f"  routes with a detected break . {n_found:,}  ({n_found / n_routes:.2%} of analysed)")
            print(f"  ... with a known throw frame . {n_known:,}  ({n_known / max(n_found, 1):.1%} of breaks)")
            print(f"  PRE-THROW breaks ............. {n_pre:,}  ({n_pre / n_routes:.2%} of analysed; "
                  f"{n_pre / max(n_known, 1):.1%} of known-throw breaks)")
            pp = pre.groupby(rd["nfl_id"]).sum()
            print(f"  pre-throw breaks per player .. mean {pp.mean():.2f}, median {pp.median():.0f}, "
                  f"max {int(pp.max())}, players with >=1: {int((pp > 0).sum())}/{n_players}, "
                  f">=2: {int((pp >= 2).sum())}")

            # -- 2. coverage by route family --------------------------------------------
            fam = pd.DataFrame({
                "routes":          rd.groupby("route_family").size(),
                "breaks":          found.groupby(rd["route_family"]).sum().astype(int),
                "pre_throw":       pre.groupby(rd["route_family"]).sum().astype(int),
                "median_depth_yd": depth.where(found).groupby(rd["route_family"]).median().round(2),
            })
            fam["break_rate"] = (fam["breaks"] / fam["routes"]).round(3)
            fam["pre_throw_rate"] = (fam["pre_throw"] / fam["routes"]).round(3)
            fam = fam.sort_values(["pre_throw", "routes"], ascending=False)
            print()
            print("coverage by route family (median depth uses detected breaks only)")
            print(fam.to_string())

            # -- declared specification curve -------------------------------------------
            sig = read_csv(RD_SIG)
            if len(sig) and {"break_angle_deg", "break_window_s"} <= set(sig.columns):
                prim = sig[is_true(sig["primary"])] if "primary" in sig.columns else sig.head(1)
                if len(prim):
                    print()
                    print(f"declared curve: {len(sig)} points on disk; primary = "
                          f"{to_num(prim['break_angle_deg']).iloc[0]:g} deg / "
                          f"{to_num(prim['break_window_s']).iloc[0]:g} s")

            # -- provenance + cross-check against the stage summary ----------------------
            print()
            print(f"provenance of the depth table: {prov_rd or 'absent'}")
            if cov:
                agree = int(cov.get("n_break_found", -1)) == n_found
                print(f"cross-check vs route_depth_summary.json: n_break_found "
                      f"{cov.get('n_break_found')} vs csv {n_found} -> "
                      f"{'agree' if agree else 'DISAGREE (do not quote either)'}")

            # -- 3. the D63 rule, evaluated ---------------------------------------------
            fams_ok = fam.index[fam["pre_throw"] >= REQ_FAM_N].tolist()
            cond_a = n_pre >= REQ_N
            cond_b = len(fams_ok) >= REQ_FAM
            print()
            print("D63 rule, computed on this run:")
            print(f"  (a) pre-throw breaks {n_pre:,} vs required {REQ_N:,} "
                  f"-> {'PASS' if cond_a else 'FAIL'}")
            print(f"  (b) families with >= {REQ_FAM_N} pre-throw breaks: {len(fams_ok)} "
                  f"({', '.join(fams_ok) if fams_ok else 'none'}) vs required {REQ_FAM} "
                  f"-> {'PASS' if cond_b else 'FAIL'}")

            thr_prov = prov_of_json(out_path("thresholds.json"))
            decidable = (MODE == "full") and (prov_rd == "FULL") and (thr_prov == "FULL")
            if decidable:
                verdict = ("KEEP continuous break_depth_yd (option 2)" if (cond_a and cond_b)
                           else f"FALL BACK to {FALLBACK}")
                print(f"  DECISION: {verdict}")
            else:
                print("  DECISION NOT TAKEN — this run cannot decide the rule. Why:")
                if MODE != "full":
                    print(f"    - MODE={MODE!r}: sample output is not a finding and not a decision"
                          " input (TASK.md, D57)")
                if prov_rd != "FULL":
                    print(f"    - route-depth provenance is {prov_rd or 'absent'}, not FULL")
                if thr_prov != "FULL":
                    print(f"    - thresholds provenance is {thr_prov or 'absent'}, not FULL")
                print("    missing full-data inputs:")
                for f in _missing_inputs():
                    print(f"      - {f}")
                print(f"  The rule stands as written: on FULL data, >= {REQ_N:,} pre-throw breaks in"
                      f" >= {REQ_FAM} families of >= {REQ_FAM_N} keeps the continuous term; anything"
                      " less falls back to option 4.")

            print()
            if MODE != "full":
                print(f"{STAMP} — the counts above are descriptive plumbing. They are not a finding,")
                print("they are not a tuning input, and the keep/fall-back decision was not taken on them.")
            else:
                print("FULL — counts above are from the full run.")


## 14. Effort table and the primary leverage model (D65)

This section renders the two stages wired after `03_route_depth`: `04_efforts` (one row per
max-effort route) and `05_models` (the within-player leverage estimand). Every number is read from
`outputs/game/lc/efforts/` and `outputs/game/lc/models/` — none is typed here.

It reports the effort-table audit (efforts per leverage bin, players per bin, per-player counts,
and how many players have efforts in **both** extremes) and the model result (leverage coefficient
with its player-clustered CI, which convergence fallback was used, the achieved analytic MDE).
`break_depth_yd` is printed once, explicitly labelled **a control, not a finding**.


In [ ]:
# ===================== 14. Effort table + primary leverage model (D65) =====================
# Reads outputs/game/lc/efforts/ and outputs/game/lc/models/ only. No number is typed here.
print("effort table and within-player leverage model (D65)")
print("=" * 88)

EFF_DIR = out_path("game", "lc", "efforts")
MOD_DIR = out_path("game", "lc", "models")
EFF_CSV = EFF_DIR / "efforts.csv"
EFF_SUM = EFF_DIR / "efforts_summary.json"
MOD_JSON = MOD_DIR / "model_result.json"
MOD_COEF = MOD_DIR / "model_coefficients.csv"

STAMP = "UNVALIDATED SAMPLE OUTPUT"

if not EFF_CSV.exists() or not EFF_SUM.exists():
    print("NOT RUN — outputs/game/lc/efforts/ is absent.")
    print("  Run 04_efforts (it needs 03_route_depth's table and the tracking parquet mirrors).")
elif not MOD_JSON.exists():
    print("EFFORT TABLE PRESENT — model stage has not run.")
    print("  Run 05_models; nothing is estimated here.")
else:
    eff = pd.read_csv(EFF_CSV, low_memory=False)
    es = read_json(EFF_SUM) or {}
    mr = read_json(MOD_JSON) or {}
    coef = read_csv(MOD_COEF)

    prov_eff = str(es.get("provenance", ""))
    prov_mod = str(mr.get("provenance", ""))
    LABEL = "FULL" if (prov_eff == "FULL" and prov_mod == "FULL") else STAMP
    print(f"label  : {LABEL}")
    print(f"source : {rel(EFF_DIR)} and {rel(MOD_DIR)}")

    # ---- 1. effort-table audit (counts only) --------------------------------------
    print()
    print("effort-table audit (descriptive; not a finding)")
    print(f"  efforts ...................... {len(eff):,} across {eff['nfl_id'].nunique()} "
          f"players and {eff['game_id'].nunique():,} games")
    print(f"  route-snap key table ......... {es.get('n_route_key_rows', 0):,} rows, "
          f"{es.get('n_route_key_players', 0)} players, {es.get('n_route_key_games', 0)} games")
    print(f"  dropped: no tracking {es.get('n_without_tracking', 0):,} · no depth "
          f"{es.get('n_without_depth', 0):,} · effort gate "
          f"{es.get('n_dropped_by_effort_gate', 0):,} · missing pre-snap WP "
          f"{es.get('n_missing_wp', 0):,}")

    print()
    print("efforts per leverage bin")
    bin_tab = (eff.groupby("leverage_bin")["nfl_id"]
                  .agg(efforts="size", players="nunique"))
    min_bin = int(es.get("min_efforts_per_bin", 0))
    per_bin_players = (eff.groupby(["leverage_bin", "nfl_id"]).size()
                          .groupby("leverage_bin")
                          .apply(lambda s: int((s >= min_bin).sum())))
    bin_tab[f"players >= {min_bin}"] = per_bin_players.reindex(bin_tab.index).fillna(0).astype(int)
    print(bin_tab.to_string())

    # both extremes: recomputed from the table, cross-checked against the summary
    pivot = (eff[eff["leverage_bin"].isin(["low", "high"])]
             .groupby(["nfl_id", "leverage_bin"]).size().unstack(fill_value=0))
    both = int(((pivot.get("low", 0) >= min_bin) & (pivot.get("high", 0) >= min_bin)).sum())         if {"low", "high"} <= set(pivot.columns) else 0
    print()
    print(f"  players with >= {min_bin} efforts in BOTH low and high leverage: {both} "
          f"(summary says {es.get('n_players_both_extremes')})")

    per_player = eff.groupby("nfl_id").size()
    min_player = int(es.get("min_efforts_per_player", 0))
    print()
    print("per-player effort counts")
    print(f"  mean {per_player.mean():.1f} · median {per_player.median():.0f} · "
          f"min {int(per_player.min())} · max {int(per_player.max())} · "
          f"p10 {per_player.quantile(0.10):.0f} · p90 {per_player.quantile(0.90):.0f}")
    print(f"  below {min_player} efforts: {int((per_player < min_player).sum())} players · "
          f"single-effort players: {int((per_player == 1).sum())} · "
          f"one-leverage-bin players: {es.get('edge_cases', {}).get('n_players_one_leverage_bin', 0)}")

    # ---- 2. the model result ------------------------------------------------------
    print()
    print("primary within-player model (estimand = leverage coefficient)")
    print(f"  specification: {mr.get('specification')}")
    print(f"  random effects: {mr.get('random_effects')}")
    print(f"  efforts {mr.get('n_efforts'):,} · players {mr.get('n_players')} · "
          f"efforts/player {float(mr.get('efforts_per_player', 0)):.1f}")
    print(f"  fallback used: {mr.get('fallback_used')}")
    for step in mr.get("ladder", []):
        print(f"    rung {step.get('rung'):<20} {step.get('status'):<7} "
              f"{step.get('reason', '')}")
    b = mr.get("beta"); lo = mr.get("ci_low"); hi = mr.get("ci_high"); pv = mr.get("p")
    print()
    print(f"  leverage coefficient {b:+.4f} log-speed per 1 within-player SD "
          f"(95% CI [{lo:+.4f}, {hi:+.4f}], p = {pv:.4f})" if all(
              isinstance(v, (int, float)) for v in (b, lo, hi, pv))
          else "  leverage coefficient unavailable (see fallback ladder)")
    mde = mr.get("mde") or {}
    print(f"  achieved analytic MDE at {float(mde.get('efforts_per_player', 0)):.1f} "
          f"efforts/player: {float(mde.get('mde', float('nan'))):.4f} log-speed "
          f"(D60: {float(mde.get('mde_at_configured', float('nan'))):.4f} at "
          f"{float(mde.get('configured_efforts_per_player', 0)):.0f}; smallest effect "
          f"of interest {float(mde.get('smallest_effect_of_interest', float('nan'))):.2f})")
    print("  the MDE is stated, not a gate: it does not change the estimate or the decision")

    # ---- 3. depth is a control ----------------------------------------------------
    d = mr.get("depth_control") or {}
    print()
    print("depth is a CONTROL, not a finding")
    print(f"  centred break_depth_yd {d.get('depth_coefficient')} "
          f"(SE {d.get('depth_se')}); break_found {d.get('break_found_coefficient')} "
          f"(SE {d.get('break_found_se')})")
    print("  reported for completeness; it is not interpreted and it is not a result")

    print()
    if LABEL != "FULL":
        print(f"{STAMP} — the numbers above are plumbing. They are not a finding, they are")
        print("not a tuning input, and they must not be quoted as a result.")
    else:
        print("FULL — the numbers above are from the full run.")


## 15. Phase 5 validation — permutation, confound, placebos, garbage time (D66)

This section renders `06_validation` — the Phase 5 checks on the primary fit. Every number is read
from `outputs/game/lc/validation/`; none is typed here, and the model is not re-specified.

It prints:

1. **Leverage permutation** — the observed statistic against its null distribution, for both declared
   shuffle schemes (within player-game, and within player-route-family), with the number of blocks
   and the share of rows that sit in a block too small to shuffle.
2. **Time-and-load orthogonalisation** — naive versus orthogonalised estimate, the attenuation, the
   within-player R² of leverage on the time/load controls, and whether the effect survives.
3. **Placebos** — leverage predicting break depth, the break indicator and time to throw.
4. **Garbage time** — the frozen quarter-and-margin rule, the counts, and the fit with and without.
5. **The declared sensitivity curve** — one row per spec, with the scale of each contrast labelled.
6. **The equivalence verdict** — TOST against the configured margin, and the separate question of
   whether the interval rules out the smallest effect of interest (an informative null).

**Sample mode never decides anything.** With `MODE="sample"` every number is stamped
`UNVALIDATED SAMPLE OUTPUT` in the artefacts and on screen. The section still runs, because the point
of sample mode is to prove the plumbing; it is not evidence about receivers.


In [ ]:
# ===================== 15. Phase 5 validation of the primary fit (D66) ====================
# Reads outputs/game/lc/validation/ only. No number is typed here.
print("Phase 5 validation — permutation, confound, placebos, garbage time (D66)")
print("=" * 88)

VAL_DIR = out_path("game", "lc", "validation")
VAL_JSON = VAL_DIR / "validation_result.json"
VAL_NULL = VAL_DIR / "permutation_null.csv"
VAL_SENS = VAL_DIR / "sensitivity_curve.csv"

STAMP = "UNVALIDATED SAMPLE OUTPUT"

if not VAL_JSON.exists():
    print("NOT RUN — outputs/game/lc/validation/validation_result.json is absent.")
    print("  Run 06_validation (it needs 04_efforts and the 05_models result).")
else:
    vr = read_json(VAL_JSON) or {}
    LABEL = "FULL" if str(vr.get("provenance", "")) == "FULL" else STAMP
    sizes = vr.get("sample_sizes") or {}
    spec = vr.get("specification_under_test") or {}
    primary = spec.get("primary_model_result") or {}

    print(f"label  : {LABEL}")
    print(f"source : {rel(VAL_DIR)}")
    print(f"model  : {spec.get('outcome')} ~ {spec.get('leverage_column')} + "
          f"{len(spec.get('controls') or [])} controls (unchanged from 05_models)")
    print(f"scope  : {sizes.get('n_efforts'):,} efforts · {sizes.get('n_players')} players · "
          f"{float(sizes.get('efforts_per_player') or 0):.1f} efforts/player")
    mde = sizes.get("mde") or {}
    print(f"MDE    : {float(mde.get('mde', float('nan'))):.4f} log-speed achieved "
          f"(smallest effect of interest {mde.get('smallest_effect_of_interest')}); stated, not a gate")
    if LABEL != "FULL":
        print(f"\n{STAMP} — everything below is plumbing, not a finding.")

    # ---- 1. permutation ---------------------------------------------------------
    perm = vr.get("permutation") or {}
    print()
    print(f"1. leverage permutation — {perm.get('reps')} shuffles per scheme")
    print(f"   rule: {perm.get('p_value_rule')}")
    schemes = perm.get("schemes") or {}
    if not schemes:
        print("   no scheme ran")
    for name, row in schemes.items():
        q = row.get("quantiles") or {}
        p = row.get("p_value")
        ptxt = f"{p:.4f}" if isinstance(p, (int, float)) else "NA"
        print(f"   {name:<14} observed {row.get('observed_beta'):+.4f} "
              f"(SE {row.get('observed_se'):.4f}) · p = {ptxt}")
        print(f"   {'':<14} null mean {row.get('null_mean'):+.4f}, sd {row.get('null_sd'):.4f}, "
              f"2.5% {q.get('q025'):+.4f}, 97.5% {q.get('q975'):+.4f} · "
              f"{row.get('blocks')} blocks, {row.get('singleton_rows')} rows in singleton blocks")
    if VAL_NULL.exists():
        nulls = pd.read_csv(VAL_NULL)
        print(f"   full null distribution on disk: {len(nulls):,} rows "
              f"({', '.join(sorted(nulls['scheme'].astype(str).unique()))})")

    # ---- 2. orthogonalisation ---------------------------------------------------
    orth = vr.get("orthogonalisation") or {}
    print()
    print("2. time-and-load orthogonalisation")
    naive = orth.get("naive") or {}
    orthf = orth.get("orthogonalised") or {}
    print(f"   naive          {naive.get('beta'):+.4f} (SE {naive.get('se'):.4f}, "
          f"p {naive.get('p'):.4f})")
    print(f"   orthogonalised {orthf.get('beta'):+.4f} (SE {orthf.get('se'):.4f}, "
          f"p {orthf.get('p'):.4f})")
    att = orth.get("attenuation_fraction")
    print(f"   attenuation    {att:.1%} of the naive magnitude" if isinstance(att, (int, float))
          else "   attenuation    NA")
    print(f"   within-player R2 of leverage on {orth.get('orthogonalised_on')}: "
          f"{orth.get('r2_leverage_on_time_load_within_player'):.3f}")
    print(f"   within-player corr with time {orth.get('within_player_corr_time'):+.3f}, "
          f"load {orth.get('within_player_corr_load'):+.3f}")
    print(f"   effect survives: {orth.get('effect_survives')} — {orth.get('note')}")

    # ---- 3. placebos ------------------------------------------------------------
    placebo = vr.get("placebo") or {}
    print()
    print("3. placebos — leverage should not predict these")
    checks = placebo.get("checks") or []
    if checks:
        rows = [{"outcome": c.get("outcome"), "status": c.get("status"),
                 "beta": c.get("beta"), "se": c.get("se"), "p": c.get("p"), "n": c.get("n")}
                for c in checks]
        print(pd.DataFrame(rows).to_string(index=False))
    for item in placebo.get("not_available") or []:
        print(f"   not available: {item.get('check')} — {item.get('reason')}")

    # ---- 4. garbage time --------------------------------------------------------
    gt = vr.get("garbage_time") or {}
    print()
    print("4. garbage time (rule frozen in leverage.garbage_time)")
    if gt.get("status") == "ok":
        rule = gt.get("rule") or {}
        print(f"   rule: |margin| >= {rule.get('q4_margin')} from Q{rule.get('q4')}, or "
              f"|margin| >= {rule.get('any_quarter_margin')} in any quarter")
        print(f"   {gt.get('n_garbage_time'):,} of {gt.get('n_total'):,} efforts "
              f"({float(gt.get('share_garbage_time') or 0):.2%})")
        shares = gt.get("share_by_leverage_bin") or {}
        if shares:
            print("   share of each leverage bin that is garbage time: " +
                  ", ".join(f"{k} {float(v):.2%}" for k, v in shares.items()))
        for key in ("with_garbage_time", "without_garbage_time"):
            fit = gt.get(key) or {}
            print(f"   {key:<21} beta {fit.get('beta'):+.4f} "
                  f"(SE {fit.get('se'):.4f}) · n {fit.get('n'):,}" if fit.get("n")
                  else f"   {key:<21} not estimable")
    else:
        print(f"   not evaluable: {gt.get('status')}")

    # ---- 5. sensitivity curve ---------------------------------------------------
    print()
    print("5. declared sensitivity curve")
    sens = vr.get("sensitivity") or {}
    rows = sens.get("rows") or []
    if rows:
        table = pd.DataFrame(rows)[["spec", "status", "comparison", "beta", "se", "p", "n"]]
        print(table.to_string(index=False))
        print("   a high-vs-low row is a bin difference; raw leverage is per unit of 4p(1-p);")
        print("   neither is on the primary per-SD scale — the CSV labels the comparison.")

    # ---- 6. equivalence ---------------------------------------------------------
    eq = vr.get("equivalence") or {}
    print()
    print("6. equivalence test")
    print(f"   margin {eq.get('margin')} · smallest effect of interest "
          f"{eq.get('smallest_effect_of_interest')} · alpha {eq.get('alpha')}")
    print(f"   estimate {eq.get('beta'):+.4f} (SE {eq.get('se'):.4f}), 95% CI "
          f"[{eq.get('ci_low'):+.4f}, {eq.get('ci_high'):+.4f}], TOST p = {eq.get('p_tost'):.4f}"
          if isinstance(eq.get("p_tost"), (int, float)) else
          f"   equivalence not tested: {eq.get('status')}")
    print(f"   equivalent at the margin: {eq.get('equivalent_at_margin')}")
    print(f"   VERDICT: {eq.get('verdict')}")

    # ---- 7. edge cases + not wired ---------------------------------------------
    edge = vr.get("edge_cases") or {}
    print()
    print("7. edge cases (reported, never crashed on)")
    print(f"   single-effort players {edge.get('n_single_effort_players')} · "
          f"single-effort player-games {edge.get('n_single_effort_player_games')} · "
          f"zero within-player leverage variance {edge.get('n_players_zero_leverage_variance')}")
    print()
    print("8. not wired this cycle (nothing below ran)")
    for item in vr.get("not_wired") or []:
        print(f"   - {item}")

    print()
    if LABEL != "FULL":
        print(f"{STAMP} — validation numbers above are plumbing. The permutation p, the")
        print("attenuation and the equivalence verdict are NOT findings until they come from FULL.")
    else:
        print("FULL — the numbers above are from the full run.")


## 16. Phase 4 remainder — matched design, headroom, ceiling bound, dose-response, slopes, TE (D67)

This section renders `07_phase4` — the Phase 4 analyses that were still outstanding after the
primary model (§14) and the Phase 5 checks (§15). Every number is read from
`outputs/game/lc/phase4/`; none is typed here, and nothing is re-specified: the stage consumes the
`04_efforts` table exactly as written.

It prints:

1. **Matched vs regression** — the coarsened exact match (within player, on route family, depth bin,
   quarter and load bin) as a high-minus-low contrast with its player-clustered bootstrap interval,
   next to the primary specification refitted on the same matched rows and converted with the
   matched-sample leverage gap. The two are labelled as different estimands on purpose.
2. **Headroom interaction** — `leverage x baseline utilization`, naive and errors-in-variables
   corrected, with the reliability ratio `lambda`; plus `leverage x vmax` and `leverage x tau`.
   Players whose low-leverage half cannot support an independent baseline are counted and named as
   *not estimable*, never silently dropped.
3. **Ceiling-bound exceedance by leverage bin** — observed and counterfactual-predicted shares of
   efforts above the plausible ceiling.
4. **Dose-response monotonicity** — the bin contrasts and the explicit monotone / not-monotone
   statement (reported as *not evaluable* if any bin contrast could not be estimated).
5. **Per-player reliability gate** — the model-based reliability against its configured threshold,
   and therefore whether a per-player slope table exists at all.
6. **TE replication** — the primary specification on tight ends with the label it is published
   under, and the n that label is based on.
7. **Checks NOT RUN** — each with the exact missing field named. Nothing is proxied.

**Sample mode never decides anything.** With `MODE="sample"` every number is stamped
`UNVALIDATED SAMPLE OUTPUT`; the section still runs because the point of sample mode is to prove the
plumbing, and it is not evidence about receivers.

In [ ]:
# ===================== 16. Phase 4 remainder (D67) ====================
# Reads outputs/game/lc/phase4/ only. No number is typed here.
print("Phase 4 remainder — matched, headroom, ceiling bound, dose, slopes, TE (D67)")
print("=" * 88)

P4_DIR  = out_path("game", "lc", "phase4")
P4_JSON = P4_DIR / "phase4_result.json"
STAMP   = "UNVALIDATED SAMPLE OUTPUT"

def num(v, digits=4, sign=False):
    try:
        if v is None or not np.isfinite(float(v)):
            return "NA"
        return f"{float(v):+.{digits}f}" if sign else f"{float(v):.{digits}f}"
    except (TypeError, ValueError):
        return "NA"

if not P4_JSON.exists():
    print("NOT RUN — outputs/game/lc/phase4/phase4_result.json is absent.")
    print("  Run 07_phase4 (it needs 04_efforts and, for the headroom and ceiling checks,")
    print("  the 02_combine_ceiling ceiling table).")
else:
    pr = read_json(P4_JSON) or {}
    LABEL = "FULL" if str(pr.get("provenance", "")) == "FULL" else STAMP
    sizes = pr.get("sample_sizes") or {}
    print(f"label  : {LABEL}")
    print(f"source : {rel(P4_DIR)}")
    print(f"scope  : {sizes.get('n_efforts'):,} efforts · {sizes.get('n_players')} players · "
          f"{sizes.get('n_games')} games")
    if LABEL != "FULL":
        print(f"\n{STAMP} — everything below is plumbing, not a finding.")

    # ---- 1. matched vs regression ------------------------------------------------
    md = pr.get("matched_design") or {}
    boot = md.get("matched") or {}
    reg = md.get("regression_on_matched") or {}
    print()
    print("1. matched design (sensitivity), within player and within exact stratum")
    print(f"   status: {md.get('status')} · {md.get('n_pairs')} pairs · "
          f"{md.get('n_players_contributing')} of {md.get('n_players_eligible')} eligible players "
          f"contribute a pair")
    print(f"   strata matched {md.get('n_strata_matched')} · one-sided strata discarded "
          f"{md.get('n_strata_one_sided')} · unmatched high/low "
          f"{md.get('n_unmatched_high')}/{md.get('n_unmatched_low')}")
    if md.get("n_players_without_pair"):
        print(f"   players with no paired cell (reported, not dropped): "
              f"{md.get('n_players_without_pair')}")
    print(f"   MATCHED    {num(boot.get('estimate'), sign=True)} log-speed "
          f"(bootstrap SE {num(boot.get('se'))}, 95% CI "
          f"[{num(boot.get('ci_low'), sign=True)}, {num(boot.get('ci_high'), sign=True)}], "
          f"n {boot.get('n_pairs')} pairs / {boot.get('n_players')} players)")
    print(f"   REGRESSION {num(reg.get('beta_per_sd'), sign=True)} per within-player SD "
          f"→ {num(reg.get('beta_high_vs_low'), sign=True)} high-minus-low "
          f"(leverage_z gap {num(reg.get('mean_leverage_z_gap'))}, "
          f"n {reg.get('n_matched_rows')} matched rows)")
    print(f"   the two are different estimands: the match is unadjusted beyond the strata, the")
    print(f"   regression carries {len(reg.get('controls') or [])} controls. {md.get('note')}")

    # ---- 2. headroom -------------------------------------------------------------
    hr = pr.get("headroom") or {}
    print()
    print("2. headroom moderation (secondary)")
    if hr.get("status") != "ok":
        print(f"   NOT RUN / not estimable: {hr.get('status')} — {hr.get('reason')}")
        if hr.get("missing_field"):
            print(f"   missing field: {hr.get('missing_field')}")
    else:
        split = hr.get("split") or {}
        print(f"   baseline: {split.get('rule')}")
        print(f"   players with a baseline {hr.get('n_players')} · excluded as not estimable "
              f"{hr.get('n_players_excluded', 0)} · baseline rows {hr.get('n_utilisation_rows')} · "
              f"comparison rows {hr.get('n_analysis_rows')}")
        for row in hr.get("rows") or []:
            print(f"   {row.get('model'):<34} {row.get('status'):<14} "
                  f"naive {num(row.get('naive_beta'), sign=True)} "
                  f"(SE {num(row.get('naive_se'))}) · lambda {num(row.get('lambda'), 3)} · "
                  f"corrected {num(row.get('corrected_beta'), sign=True)} "
                  f"(SE {num(row.get('corrected_se'))}) · n {row.get('n')} / "
                  f"{row.get('n_players')} players")
        print(f"   correction: see headroom.csv 'correction' column — "
              f"lambda = 1 - mean(SE^2)/var(moderator), corrected = naive / lambda")

    # ---- 3. ceiling bound --------------------------------------------------------
    cb = pr.get("ceiling_bound") or {}
    print()
    print("3. ceiling-bound check — can high leverage exceed the plausible ceiling?")
    if cb.get("status") != "ok":
        print(f"   NOT RUN ({cb.get('status')}) — missing field: {cb.get('missing_field')}")
        if cb.get("reason"):
            print(f"   {cb.get('reason')}")
    else:
        print(f"   exceedance defined as effort fraction > {cb.get('exceedance_ratio')} · "
              f"{cb.get('n_efforts_with_ceiling'):,} efforts carry a ceiling "
              f"({cb.get('n_efforts_without_ceiling'):,} do not)")
        print(f"   {'bin':<8}{'n':>7}{'obs exc':>10}{'pred@high exc':>15}")
        for row in cb.get("rows") or []:
            print(f"   {row.get('leverage_bin'):<8}{row.get('n'):>7}"
                  f"{num(row.get('observed_exceedance_rate'), 4):>10}"
                  f"{num(row.get('predicted_exceedance_rate_at_high'), 4):>15}")
        allrow = cb.get("all_rows") or {}
        print(f"   {'all':<8}{allrow.get('n'):>7}"
              f"{num(allrow.get('observed_exceedance_rate'), 4):>10}"
              f"{num(allrow.get('predicted_exceedance_rate_at_high'), 4):>15}")
        print(f"   max predicted fraction at high leverage: "
              f"{num(cb.get('max_predicted_fraction_at_high'))} "
              f"(observed max {num(cb.get('max_observed_fraction'))})")

    # ---- 4. dose-response --------------------------------------------------------
    dose = pr.get("dose_response") or {}
    print()
    print("4. dose-response across leverage bins")
    if dose.get("status") != "ok":
        print(f"   NOT RUN ({dose.get('status')}) — {dose.get('note')}")
    else:
        for row in dose.get("rows") or []:
            if row.get("role") == "reference":
                print(f"   {row.get('term'):<24} reference (n {row.get('n')})")
            else:
                print(f"   {row.get('term'):<24} {row.get('status', ''):<14} "
                      f"{num(row.get('beta'), sign=True)} "
                      f"(SE {num(row.get('se'))}, n {row.get('n')})")
        trend = dose.get("trend") or {}
        print(f"   monotone: {dose.get('monotone')} ({dose.get('direction')}) · "
              f"trend {num(trend.get('beta'), sign=True)} per bin "
              f"(SE {num(trend.get('se'))}, p {num(trend.get('p'), 4)})")
        if dose.get("not_estimable_bins"):
            print(f"   bins not estimable: {', '.join(dose['not_estimable_bins'])}")

    # ---- 5. reliability gate -----------------------------------------------------
    sl = pr.get("per_player_slopes") or {}
    print()
    print("5. per-player slopes — reliability gate")
    print(f"   status: {sl.get('status')} · reliability {num(sl.get('reliability'), 3)} vs "
          f"threshold {sl.get('reliability_threshold')} · eligible players "
          f"{sl.get('n_players_eligible')}")
    print(f"   rule: {sl.get('reliability_rule')}")
    print(f"   {sl.get('reason')}")
    if sl.get("status") != "reported":
        print("   no per-player table is emitted — a noisy slope is not evidence about a player")

    # ---- 6. TE replication -------------------------------------------------------
    rp = pr.get("replication") or {}
    print()
    print("6. TE replication")
    if rp.get("status") != "ok":
        print(f"   NOT RUN ({rp.get('status')}) — {rp.get('reason')} "
              f"(missing field: {rp.get('missing_field')})")
    else:
        print(f"   {rp.get('position')}: {num(rp.get('beta'), sign=True)} "
              f"(SE {num(rp.get('se'))}, 95% CI [{num(rp.get('ci_low'), sign=True)}, "
              f"{num(rp.get('ci_high'), sign=True)}]) · n {rp.get('n_efforts'):,} efforts / "
              f"{rp.get('n_players')} players")
        print(f"   LABEL: {rp.get('label')} (on {rp.get('labelled_on_n_players')} players; "
              f"threshold {rp.get('exploratory_below_players')})")

    # ---- 7. checks that could not run -------------------------------------------
    print()
    print("7. checks NOT RUN — the missing field is named, never proxied")
    notrun = pr.get("checks_not_run") or []
    if not notrun:
        print("   none")
    for row in notrun:
        print(f"   - {row.get('check')}: missing field `{row.get('missing_field')}` — "
              f"{row.get('reason')}")

    print()
    if LABEL != "FULL":
        print(f"{STAMP} — the estimates above are plumbing. The matched contrast, the headroom")
        print("interaction and the exceedance rates are NOT findings until they come from FULL.")
    else:
        print("FULL — the numbers above are from the full run.")